# Notebook 02 v5 — Deterministic Weather Experts, Ensembles, and Paired Evaluation

This notebook trains provider-specific and pooled LightGBM wind models on the
D2 and exact-run panels as separate information-time tracks. It then evaluates
static ensembles, non-negative linear stacking, forecast-origin-safe Hedge,
forecast-disagreement regimes, common-support benchmark comparisons, and
paired statistical inference.


## Methodological rules

- The D2 and exact-run panels are both used, but as **separate modelling tracks**.
- No random train-test split.
- No feature may use information published after the day-ahead forecast origin.
- Spatial point-count columns are metadata and are excluded from modelling.
- Day-ahead and imbalance-price fields are excluded from wind forecasting.
- Correlation analysis uses training data only.
- LightGBM boosting iteration and parameter configuration are selected inside the outer training period.
- Outer validation predictions are reserved for ensemble-weight estimation.
- Test observations are untouched until the expert configuration and ensemble rule are frozen.
- Hedge updates only from outcomes available by the current forecast origin.
- Dirichlet and SLSQP weights are non-negative and sum to one.
- DK1 and DK2 are estimated and reported separately.
- DST is handled through timezone-aware timestamps and observed timestamp keys.

## v5 changes

This revision preserves the v4 modelling methodology and adds the following
publication-oriented corrections:

1. exact-run outer test windows are non-overlapping;
2. LightGBM GPU configuration is recorded in the run audit;
3. climatology uses hierarchical fallbacks, so the short exact history cannot
   produce an empty benchmark;
4. residual correlations use only valid panel-specific provider experts;
5. benchmark comparisons are recomputed on identical timestamps;
6. fixed, pre-specified pairwise comparisons receive block-bootstrap and
   Newey–West uncertainty estimates;
7. D2 and exact predictions are compared only on common delivery timestamps;
8. outputs are written to a versioned directory so the completed v4 run is not
   overwritten.


In [ ]:
# ============================================================
# 1. PROJECT SETUP
# ============================================================

from __future__ import annotations

from dataclasses import dataclass, asdict
from itertools import combinations
from pathlib import Path
from statistics import NormalDist
from typing import Any, Iterable, Sequence
import hashlib
import json
import math
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# v5 is configured for the final GPU-enabled run.
RUN_MODE = "FULL"       # "AUDIT" or "FULL"
SEARCH_MODE = "FULL"    # "SMOKE" or "FULL"
STRICT_SHAPE_CHECK = True

# Keep v4 outputs immutable.
OUTPUT_RUN_NAME = "notebook_02_v5_full"

LOCAL_TIMEZONE = "Europe/Copenhagen"
PRICE_AREAS = ("DK1", "DK2")
TARGET_COLUMN = "actual_wind_mwh"
KEY_COLUMNS = ("timestamp_utc", "price_area")

# Operational forecast-origin assumption used for safe target lags
# and online Hedge updates.
FORECAST_ORIGIN_HOUR_LOCAL = 12
ACTUAL_DATA_LATENCY_HOURS = 1

D2_SOURCES = (
    "gfs",
    "icon_eu",
    "ecmwf",
    "gem_global",
    "jma_gsm",
)

EXACT_SOURCES = (
    "gfs",
    "icon_eu",
    "ecmwf",
    "dmi",
    "arpege_europe",
    "ukmo_global",
    "cma_grapes",
)

EXPECTED_D2_SHAPE = (33_648, 228)
EXPECTED_EXACT_SHAPE = (9_458, 315)

D2_FOLD_CONFIG = {
    "min_train_days": 365,
    "validation_days": 60,
    "test_days": 90,
    "step_days": 90,
}

# step_days == test_days prevents repeated exact test timestamps.
EXACT_FOLD_CONFIG = {
    "min_train_days": 90,
    "validation_days": 30,
    "test_days": 45,
    "step_days": 45,
}

LIGHTGBM_BASE_PARAMS = {
    "objective": "regression_l1",
    "learning_rate": 0.025,
    "max_depth": -1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "verbosity": -1,
    # Windows GPU backend validated by the user before this run.
    "device_type": "gpu",
    "gpu_platform_id": -1,
    "gpu_device_id": -1,
    "gpu_use_dp": False,
    "max_bin": 63,
}

# Small, prespecified inner-training candidate set.
LIGHTGBM_CANDIDATES = (
    {
        "name": "balanced",
        "num_leaves": 31,
        "min_child_samples": 40,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
    },
    {
        "name": "regularized",
        "num_leaves": 20,
        "min_child_samples": 70,
        "reg_alpha": 0.2,
        "reg_lambda": 2.0,
    },
    {
        "name": "flexible",
        "num_leaves": 63,
        "min_child_samples": 30,
        "reg_alpha": 0.0,
        "reg_lambda": 1.5,
    },
)

INNER_VALIDATION_DAYS_D2 = 45
INNER_VALIDATION_DAYS_EXACT = 21
MAX_BOOSTING_ROUNDS = 2_000
EARLY_STOPPING_ROUNDS = 100

DIRICHLET_ALPHAS = (0.3, 1.0, 3.0)
DIRICHLET_DRAWS = 30_000 if SEARCH_MODE == "FULL" else 3_000
DIRICHLET_BATCH_SIZE = 2_000
SLSQP_RANDOM_STARTS = 12 if SEARCH_MODE == "FULL" else 4

HEDGE_ETA_GRID = (0.05, 0.10, 0.25, 0.50, 1.00, 2.00)
HEDGE_SHRINKAGE_GRID = (0.00, 0.01, 0.05)
HEDGE_LOSS_CLIP = 5.0

REGIME_QUANTILES = (1.0 / 3.0, 2.0 / 3.0)
MIN_REGIME_VALIDATION_ROWS = 100

# Paired inference configuration.
PAIRWISE_ALPHA = 0.05
HAC_MAX_LAG_HOURS = 48
BLOCK_BOOTSTRAP_DRAWS = 2_000

if RUN_MODE not in {"AUDIT", "FULL"}:
    raise ValueError("RUN_MODE must be AUDIT or FULL.")

if SEARCH_MODE not in {"SMOKE", "FULL"}:
    raise ValueError("SEARCH_MODE must be SMOKE or FULL.")

print(
    {
        "run_mode": RUN_MODE,
        "search_mode": SEARCH_MODE,
        "output_run_name": OUTPUT_RUN_NAME,
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "random_seed": RANDOM_SEED,
        "lightgbm_device": LIGHTGBM_BASE_PARAMS["device_type"],
        "d2_track": D2_SOURCES,
        "exact_track": EXACT_SOURCES,
    }
)


In [ ]:
# ============================================================
# 2. LOCATE PROJECT ROOT AND VERSIONED OUTPUT DIRECTORIES
# ============================================================

def find_project_root(starting_path: Path | None = None) -> Path:
    current = Path(starting_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            candidate.joinpath("pyproject.toml").exists()
            and candidate.joinpath("data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected pyproject.toml and data/."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# v5 does not overwrite the completed v4 result set.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / OUTPUT_RUN_NAME
AUDIT_DIR = OUTPUT_DIR / "audits"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in (
    AUDIT_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    TABLE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Versioned output root:", OUTPUT_DIR)


## Frozen input contract

Notebook 02 does not reconstruct data. It consumes the canonical Parquet outputs from Notebook 01.

Because earlier revisions used more than one historical-panel filename, the resolver below checks an ordered list of candidates. The exact-run canonical filename should resolve to the strict seven-provider common panel.

In [ ]:
# ============================================================
# 3. RESOLVE CANONICAL INPUT FILES
# ============================================================

D2_PANEL_CANDIDATES = (
    PROCESSED_DIR / "wind_model_panel_d2_hourly.parquet",
    PROCESSED_DIR / "wind_model_panel_hourly.parquet",
    PROCESSED_DIR / "wind_model_panel.parquet",
    PROCESSED_DIR / "wind_expert_model_panel.parquet",
)

EXACT_PANEL_CANDIDATES = (
    PROCESSED_DIR / "wind_model_panel_exact_runs_hourly.parquet",
    PROCESSED_DIR / "wind_model_panel_exact_runs_seven_source_common_hourly.parquet",
)


def resolve_existing_path(
    candidates: Sequence[Path],
    *,
    label: str,
) -> Path:
    existing = [path for path in candidates if path.exists()]

    if not existing:
        raise FileNotFoundError(
            f"No {label} file was found. Checked: "
            + ", ".join(str(path) for path in candidates)
        )

    if len(existing) > 1:
        print(
            f"Multiple {label} files exist. Using the first candidate:",
            existing[0],
        )

    return existing[0]


D2_PANEL_PATH = resolve_existing_path(
    D2_PANEL_CANDIDATES,
    label="D2 panel",
)

EXACT_PANEL_PATH = resolve_existing_path(
    EXACT_PANEL_CANDIDATES,
    label="exact-run panel",
)

print("D2 panel:", D2_PANEL_PATH)
print("Exact-run panel:", EXACT_PANEL_PATH)

In [ ]:
# ============================================================
# 4. LOAD AND VALIDATE FROZEN PANELS
# ============================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1_048_576,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def validate_model_panel(
    frame: pd.DataFrame,
    *,
    panel_name: str,
    expected_shape: tuple[int, int],
) -> pd.DataFrame:
    required_columns = {
        "timestamp_utc",
        "price_area",
        TARGET_COLUMN,
    }

    missing_columns = sorted(required_columns.difference(frame.columns))

    if missing_columns:
        raise RuntimeError(
            f"{panel_name} is missing required columns: {missing_columns}"
        )

    if STRICT_SHAPE_CHECK and frame.shape != expected_shape:
        raise RuntimeError(
            f"{panel_name} shape mismatch: "
            f"observed={frame.shape}, expected={expected_shape}"
        )

    validated = frame.copy()

    timestamps = pd.to_datetime(
        validated["timestamp_utc"],
        errors="raise",
    )

    if timestamps.dt.tz is None:
        raise RuntimeError(
            f"{panel_name} contains timezone-naive timestamps."
        )

    validated["timestamp_utc"] = timestamps.dt.tz_convert("UTC")

    # Force the modelling target to numeric at the dataset gate.
    # Any non-numeric value becomes missing and is rejected below.
    validated[TARGET_COLUMN] = pd.to_numeric(
        validated[TARGET_COLUMN],
        errors="coerce",
    )

    duplicate_count = int(
        validated.duplicated(list(KEY_COLUMNS)).sum()
    )

    if duplicate_count:
        raise RuntimeError(
            f"{panel_name} contains {duplicate_count} duplicate keys."
        )

    observed_areas = set(
        validated["price_area"].dropna().astype(str).unique()
    )

    if observed_areas != set(PRICE_AREAS):
        raise RuntimeError(
            f"{panel_name} price-area mismatch: "
            f"observed={sorted(observed_areas)}, "
            f"expected={sorted(PRICE_AREAS)}"
        )

    missing_target_count = int(
        validated[TARGET_COLUMN].isna().sum()
    )

    if missing_target_count:
        raise RuntimeError(
            f"{panel_name} has {missing_target_count} "
            "missing or non-numeric target values."
        )

    return (
        validated.sort_values(list(KEY_COLUMNS))
        .reset_index(drop=True)
    )


d2_panel = validate_model_panel(
    pd.read_parquet(D2_PANEL_PATH),
    panel_name="D2 panel",
    expected_shape=EXPECTED_D2_SHAPE,
)

exact_panel = validate_model_panel(
    pd.read_parquet(EXACT_PANEL_PATH),
    panel_name="Exact-run panel",
    expected_shape=EXPECTED_EXACT_SHAPE,
)

print("D2 shape:", d2_panel.shape)
print("Exact shape:", exact_panel.shape)

In [ ]:
# ============================================================
# 5. SAVE INPUT DATASET MANIFEST
# ============================================================

def panel_manifest(
    frame: pd.DataFrame,
    path: Path,
) -> dict:
    return {
        "path": str(path),
        "shape": list(frame.shape),
        "sha256": sha256_file(path),
        "first_timestamp_utc": str(frame["timestamp_utc"].min()),
        "last_timestamp_utc": str(frame["timestamp_utc"].max()),
        "price_areas": sorted(
            frame["price_area"].dropna().astype(str).unique().tolist()
        ),
        "target_column": TARGET_COLUMN,
    }


dataset_manifest = {
    "d2": panel_manifest(d2_panel, D2_PANEL_PATH),
    "exact": panel_manifest(exact_panel, EXACT_PANEL_PATH),
}

manifest_path = (
    AUDIT_DIR
    / "notebook_02_input_dataset_manifest.json"
)

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(dataset_manifest, file, indent=2)

display(
    pd.DataFrame(
        [
            {
                "panel": name,
                "rows": values["shape"][0],
                "columns": values["shape"][1],
                "first_timestamp_utc": values["first_timestamp_utc"],
                "last_timestamp_utc": values["last_timestamp_utc"],
                "sha256": values["sha256"][:16] + "...",
            }
            for name, values in dataset_manifest.items()
        ]
    )
)

## Decision-time feature registry

Every column receives one of the following statuses:

- `allowed`: eligible model input.
- `identifier`: key or descriptive field, not a predictor.
- `target`: realized wind-generation target.
- `benchmark_only`: retained for comparison but excluded from provider experts.
- `post_outcome`: only known after delivery or settlement.
- `manual_review`: ambiguous availability or semantics.

The automatic rules are conservative. Any `manual_review` item must be resolved explicitly before `RUN_MODE = "FULL"`.

In [ ]:
# ============================================================
# 6. COLUMN CLASSIFICATION UTILITIES
# ============================================================

IDENTIFIER_COLUMNS = {
    "timestamp_utc",
    "price_area",
    "target_date_local",
    "date_local",
    "local_date",
    "delivery_date",
    "model_alias",
    "api_model",
    "provider",
}

TARGET_PATTERNS = (
    r"^actual_wind",
    r"realized_wind",
    r"observed_wind",
)

BENCHMARK_PATTERNS = (
    r"energinet.*forecast",
    r"public.*forecast",
)

POST_OUTCOME_PATTERNS = (
    r"imbalance",
    r"regulating",
    r"settlement",
    r"realized_price",
    r"forecast_error",
    r"residual",
    r"actual_",
)

PRICE_PATTERNS = (
    r"day_ahead",
    r"elspot",
    r"spot_price",
    r"price_eur",
    r"price_dkk",
)

SAFE_SHARED_PATTERNS = (
    r"hour_",
    r"day_of_week",
    r"month",
    r"is_weekend",
    r"sin",
    r"cos",
    r"lag_",
    r"installed_capacity",
)

WEATHER_PATTERNS = (
    r"wind_speed",
    r"wind_direction",
    r"wind_gust",
    r"temperature",
    r"pressure",
    r"cloud",
    r"precip",
    r"humidity",
    r"cape",
    r"radiation",
)


def matches_any(
    value: str,
    patterns: Sequence[str],
) -> bool:
    return any(
        re.search(pattern, value, flags=re.IGNORECASE)
        for pattern in patterns
    )


def infer_provider(
    column: str,
    providers: Sequence[str],
) -> str:
    normalized = column.lower()

    for provider in sorted(providers, key=len, reverse=True):
        if provider.lower() in normalized:
            return provider

    return ""


def classify_column(
    column: str,
    dtype: str,
    providers: Sequence[str],
) -> dict:
    lower = column.lower()
    provider = infer_provider(column, providers)

    if column in IDENTIFIER_COLUMNS:
        status = "identifier"
        reason = "Key or descriptive field."
        group = "identifier"
    elif column == TARGET_COLUMN or matches_any(lower, TARGET_PATTERNS):
        status = "target"
        reason = "Realized wind-generation outcome."
        group = "target"
    elif matches_any(lower, BENCHMARK_PATTERNS):
        status = "benchmark_only"
        reason = "Public benchmark retained for evaluation."
        group = "benchmark"
    elif matches_any(lower, POST_OUTCOME_PATTERNS):
        status = "post_outcome"
        reason = "Potentially known only after delivery or settlement."
        group = "post_outcome"
    elif matches_any(lower, PRICE_PATTERNS):
        status = "manual_review"
        reason = (
            "Price availability depends on the precise nomination decision time; "
            "excluded from wind experts by default."
        )
        group = "price"
    elif provider and matches_any(lower, WEATHER_PATTERNS):
        status = "allowed"
        reason = "Provider weather feature."
        group = "provider_weather"
    elif matches_any(lower, SAFE_SHARED_PATTERNS):
        status = "allowed"
        reason = "Calendar, capacity, or valid lag feature."
        group = "shared"
    elif pd.api.types.is_numeric_dtype(dtype):
        status = "manual_review"
        reason = "Numeric feature requires semantic review."
        group = "other_numeric"
    else:
        status = "identifier"
        reason = "Non-numeric field excluded from default model matrix."
        group = "other"

    return {
        "feature": column,
        "dtype": str(dtype),
        "group": group,
        "provider": provider,
        "decision_time_status": status,
        "reason": reason,
        "include_default": status == "allowed",
    }

In [ ]:
# ============================================================
# 7. BUILD FEATURE REGISTRIES
# ============================================================

def build_feature_registry(
    frame: pd.DataFrame,
    providers: Sequence[str],
    panel_name: str,
) -> pd.DataFrame:
    registry = pd.DataFrame(
        [
            classify_column(
                column,
                frame[column].dtype,
                providers,
            )
            for column in frame.columns
        ]
    )

    registry.insert(0, "panel", panel_name)
    return registry


d2_feature_registry = build_feature_registry(
    d2_panel,
    D2_SOURCES,
    "d2",
)

exact_feature_registry = build_feature_registry(
    exact_panel,
    EXACT_SOURCES,
    "exact",
)

feature_registry = pd.concat(
    [d2_feature_registry, exact_feature_registry],
    ignore_index=True,
)

feature_registry.to_csv(
    AUDIT_DIR / "notebook_02_feature_registry.csv",
    index=False,
)

display(
    feature_registry.groupby(
        ["panel", "decision_time_status", "group"],
        as_index=False,
    ).size()
)

manual_review_features = feature_registry.loc[
    feature_registry["decision_time_status"].eq("manual_review")
].copy()

print("Manual-review feature count:", len(manual_review_features))
display(manual_review_features.head(100))

### Reviewed feature-availability overrides

The previously ambiguous columns have now been classified:

- `__points_all`, `__points_land`, and `__points_offshore` are spatial-support metadata and are excluded.
- `day_ahead_price_eur_mwh` and `day_ahead_intervals_per_hour` are retained for downstream commercial work but excluded from wind forecasting.

The exact-run weather variables remain eligible. This review does **not** remove the exact-run panel.

In [ ]:
# ============================================================
# 8. REVIEWED FEATURE-AVAILABILITY OVERRIDES
# ============================================================

D2_FEATURE_OVERRIDES: dict[str, str] = {
    "day_ahead_intervals_per_hour": "benchmark_only",
    "day_ahead_price_eur_mwh": "benchmark_only",
    "ecmwf__points_all": "identifier",
    "ecmwf__points_land": "identifier",
    "ecmwf__points_offshore": "identifier",
    "gem_global__points_all": "identifier",
    "gem_global__points_land": "identifier",
    "gem_global__points_offshore": "identifier",
    "gfs__points_all": "identifier",
    "gfs__points_land": "identifier",
    "gfs__points_offshore": "identifier",
    "icon_eu__points_all": "identifier",
    "icon_eu__points_land": "identifier",
    "icon_eu__points_offshore": "identifier",
    "jma_gsm__points_all": "identifier",
    "jma_gsm__points_land": "identifier",
    "jma_gsm__points_offshore": "identifier",
}

EXACT_FEATURE_OVERRIDES: dict[str, str] = {
    "arpege_europe__points_all": "identifier",
    "arpege_europe__points_land": "identifier",
    "arpege_europe__points_offshore": "identifier",
    "cma_grapes__points_all": "identifier",
    "cma_grapes__points_land": "identifier",
    "cma_grapes__points_offshore": "identifier",
    "day_ahead_intervals_per_hour": "benchmark_only",
    "day_ahead_price_eur_mwh": "benchmark_only",
    "dmi__points_all": "identifier",
    "dmi__points_land": "identifier",
    "dmi__points_offshore": "identifier",
    "ecmwf__points_all": "identifier",
    "ecmwf__points_land": "identifier",
    "ecmwf__points_offshore": "identifier",
    "gfs__points_all": "identifier",
    "gfs__points_land": "identifier",
    "gfs__points_offshore": "identifier",
    "icon_eu__points_all": "identifier",
    "icon_eu__points_land": "identifier",
    "icon_eu__points_offshore": "identifier",
    "ukmo_global__points_all": "identifier",
    "ukmo_global__points_land": "identifier",
    "ukmo_global__points_offshore": "identifier",
}


def apply_feature_overrides(
    registry: pd.DataFrame,
    overrides: dict[str, str],
) -> pd.DataFrame:
    updated = registry.copy()

    valid_statuses = {
        "allowed",
        "identifier",
        "target",
        "benchmark_only",
        "post_outcome",
        "manual_review",
    }

    invalid = sorted(set(overrides.values()).difference(valid_statuses))
    if invalid:
        raise ValueError(f"Invalid override statuses: {invalid}")

    for feature, status in overrides.items():
        mask = updated["feature"].eq(feature)

        if not mask.any():
            raise KeyError(
                f"Override feature {feature!r} was not found."
            )

        updated.loc[mask, "decision_time_status"] = status
        updated.loc[mask, "include_default"] = status == "allowed"
        updated.loc[mask, "reason"] = "Reviewed Notebook 02 override."

    return updated


d2_feature_registry = apply_feature_overrides(
    d2_feature_registry,
    D2_FEATURE_OVERRIDES,
)

exact_feature_registry = apply_feature_overrides(
    exact_feature_registry,
    EXACT_FEATURE_OVERRIDES,
)

reviewed_feature_registry = pd.concat(
    [d2_feature_registry, exact_feature_registry],
    ignore_index=True,
)

reviewed_feature_registry.to_csv(
    AUDIT_DIR / "notebook_02_feature_registry_reviewed.csv",
    index=False,
)

remaining_manual_review = reviewed_feature_registry.loc[
    reviewed_feature_registry[
        "decision_time_status"
    ].eq("manual_review")
]

print(
    "Remaining manual-review features:",
    len(remaining_manual_review),
)

if len(remaining_manual_review) != 0:
    display(remaining_manual_review)
    raise RuntimeError(
        "Feature review is incomplete."
    )

## Time-aware folds

The fold builder uses local delivery dates and expanding training windows. It never shuffles rows.

Each fold contains:

- an expanding training period;
- a contiguous validation period;
- a contiguous test period;
- no overlap among train, validation, and test dates.

The exact-run sample is shorter, so its fold lengths are deliberately smaller.

In [ ]:
# ============================================================
# 9. EXPANDING FOLD DEFINITIONS
# ============================================================

@dataclass(frozen=True)
class TemporalFold:
    panel: str
    fold_id: int
    train_start: str
    train_end: str
    validation_start: str
    validation_end: str
    test_start: str
    test_end: str


def local_dates(frame: pd.DataFrame) -> pd.Series:
    return (
        frame["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.normalize()
    )


def make_expanding_folds(
    frame: pd.DataFrame,
    *,
    panel_name: str,
    min_train_days: int,
    validation_days: int,
    test_days: int,
    step_days: int,
) -> list[TemporalFold]:
    unique_dates = pd.DatetimeIndex(
        local_dates(frame).drop_duplicates().sort_values()
    )

    folds: list[TemporalFold] = []
    fold_id = 1
    cursor = min_train_days

    while (
        cursor + validation_days + test_days
        <= len(unique_dates)
    ):
        train_dates = unique_dates[:cursor]
        validation_slice = unique_dates[
            cursor : cursor + validation_days
        ]
        test_slice = unique_dates[
            cursor + validation_days :
            cursor + validation_days + test_days
        ]

        folds.append(
            TemporalFold(
                panel=panel_name,
                fold_id=fold_id,
                train_start=str(train_dates[0].date()),
                train_end=str(train_dates[-1].date()),
                validation_start=str(validation_slice[0].date()),
                validation_end=str(validation_slice[-1].date()),
                test_start=str(test_slice[0].date()),
                test_end=str(test_slice[-1].date()),
            )
        )

        cursor += step_days
        fold_id += 1

    if not folds:
        raise RuntimeError(
            f"No folds could be created for {panel_name}. "
            "Reduce fold lengths or inspect temporal support."
        )

    return folds


d2_folds = make_expanding_folds(
    d2_panel,
    panel_name="d2",
    **D2_FOLD_CONFIG,
)

exact_folds = make_expanding_folds(
    exact_panel,
    panel_name="exact",
    **EXACT_FOLD_CONFIG,
)

temporal_folds = pd.DataFrame(
    [asdict(fold) for fold in (*d2_folds, *exact_folds)]
)

temporal_folds.to_csv(
    AUDIT_DIR / "notebook_02_temporal_folds.csv",
    index=False,
)

display(temporal_folds)

In [ ]:
# ============================================================
# 10. FOLD MASKS AND INTEGRITY CHECKS
# ============================================================

def fold_masks(
    frame: pd.DataFrame,
    fold: TemporalFold,
) -> dict[str, pd.Series]:
    dates = local_dates(frame)

    train_start = pd.Timestamp(fold.train_start, tz=LOCAL_TIMEZONE)
    train_end = pd.Timestamp(fold.train_end, tz=LOCAL_TIMEZONE)

    validation_start = pd.Timestamp(
        fold.validation_start,
        tz=LOCAL_TIMEZONE,
    )
    validation_end = pd.Timestamp(
        fold.validation_end,
        tz=LOCAL_TIMEZONE,
    )

    test_start = pd.Timestamp(fold.test_start, tz=LOCAL_TIMEZONE)
    test_end = pd.Timestamp(fold.test_end, tz=LOCAL_TIMEZONE)

    masks = {
        "train": dates.between(train_start, train_end),
        "validation": dates.between(
            validation_start,
            validation_end,
        ),
        "test": dates.between(test_start, test_end),
    }

    overlap = (
        masks["train"].astype(int)
        + masks["validation"].astype(int)
        + masks["test"].astype(int)
    )

    if int((overlap > 1).sum()) != 0:
        raise RuntimeError(
            f"Fold {fold.panel}-{fold.fold_id} contains overlapping rows."
        )

    return masks


fold_integrity_rows = []

for panel_name, frame, folds in (
    ("d2", d2_panel, d2_folds),
    ("exact", exact_panel, exact_folds),
):
    for fold in folds:
        masks = fold_masks(frame, fold)

        for split_name, mask in masks.items():
            fold_integrity_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold.fold_id,
                    "split": split_name,
                    "rows": int(mask.sum()),
                    "unique_dates": int(
                        local_dates(frame.loc[mask]).nunique()
                    ),
                }
            )

fold_integrity = pd.DataFrame(fold_integrity_rows)
display(fold_integrity)

if (fold_integrity["rows"] == 0).any():
    raise RuntimeError("At least one fold split is empty.")

In [ ]:
# ============================================================
# 10A. CROSS-FOLD TEST-WINDOW OVERLAP AUDIT
# ============================================================

def test_membership_table(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    *,
    panel_name: str,
) -> pd.DataFrame:
    blocks = []

    for fold in folds:
        mask = fold_masks(frame, fold)["test"]

        block = frame.loc[
            mask,
            ["timestamp_utc", "price_area"],
        ].copy()

        block["panel"] = panel_name
        block["fold_id"] = fold.fold_id
        blocks.append(block)

    return pd.concat(blocks, ignore_index=True)


test_membership = pd.concat(
    [
        test_membership_table(
            d2_panel,
            d2_folds,
            panel_name="d2",
        ),
        test_membership_table(
            exact_panel,
            exact_folds,
            panel_name="exact",
        ),
    ],
    ignore_index=True,
)

test_key_counts = (
    test_membership.groupby(
        ["panel", "timestamp_utc", "price_area"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "fold_memberships"})
)

test_overlap_summary = (
    test_key_counts.groupby("panel", as_index=False)
    .agg(
        unique_test_keys=("fold_memberships", "size"),
        maximum_fold_memberships=("fold_memberships", "max"),
        overlapping_test_keys=(
            "fold_memberships",
            lambda values: int((values > 1).sum()),
        ),
    )
)

test_overlap_summary.to_csv(
    AUDIT_DIR / "notebook_02_test_window_overlap_audit.csv",
    index=False,
)

display(test_overlap_summary)

if int(test_overlap_summary["overlapping_test_keys"].sum()) != 0:
    raise RuntimeError(
        "Outer test windows overlap across folds. "
        "Use step_days >= test_days or explicitly redesign inference."
    )


## Exploratory dependence analysis

The analysis below is performed on training periods only.

Required views:

- Pearson and Spearman feature-target correlation;
- provider-to-provider weather correlation;
- provider prediction residual correlation after the expert models exist;
- stability of correlation signs and ranks across folds;
- relationships by area, season, hour, wind regime, and provider-disagreement regime.

Correlation is descriptive, not a causal or automatic feature-selection rule.

In [ ]:
# ============================================================
# 11. TRAINING-ONLY FEATURE-TARGET CORRELATIONS
# ============================================================

def allowed_numeric_features(
    frame: pd.DataFrame,
    registry: pd.DataFrame,
) -> list[str]:
    allowed = registry.loc[
        registry["decision_time_status"].eq("allowed"),
        "feature",
    ].tolist()

    return [
        column
        for column in allowed
        if (
            column in frame.columns
            and pd.api.types.is_numeric_dtype(frame[column])
        )
    ]


def feature_target_correlations(
    frame: pd.DataFrame,
    registry: pd.DataFrame,
    folds: Sequence[TemporalFold],
    *,
    panel_name: str,
) -> pd.DataFrame:
    feature_columns = allowed_numeric_features(frame, registry)
    rows: list[dict] = []

    for fold in folds:
        masks = fold_masks(frame, fold)

        for price_area in PRICE_AREAS:
            area_train = frame.loc[
                masks["train"]
                & frame["price_area"].eq(price_area),
                [TARGET_COLUMN, *feature_columns],
            ]

            for feature in feature_columns:
                pair = area_train[[feature, TARGET_COLUMN]].dropna()

                if len(pair) < 50 or pair[feature].nunique() < 2:
                    continue

                rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold.fold_id,
                        "price_area": price_area,
                        "feature": feature,
                        "pearson": pair[feature].corr(
                            pair[TARGET_COLUMN],
                            method="pearson",
                        ),
                        "spearman": pair[feature].corr(
                            pair[TARGET_COLUMN],
                            method="spearman",
                        ),
                        "observations": len(pair),
                    }
                )

    return pd.DataFrame(rows)


d2_correlations = feature_target_correlations(
    d2_panel,
    d2_feature_registry,
    d2_folds,
    panel_name="d2",
)

exact_correlations = feature_target_correlations(
    exact_panel,
    exact_feature_registry,
    exact_folds,
    panel_name="exact",
)

feature_correlations = pd.concat(
    [d2_correlations, exact_correlations],
    ignore_index=True,
)

feature_correlations.to_parquet(
    AUDIT_DIR / "notebook_02_feature_target_correlations.parquet",
    index=False,
)

display(
    feature_correlations.assign(
        absolute_spearman=lambda frame: frame["spearman"].abs()
    )
    .sort_values(
        ["panel", "price_area", "absolute_spearman"],
        ascending=[True, True, False],
    )
    .groupby(["panel", "price_area"])
    .head(20)
)

In [ ]:
# ============================================================
# 12. CORRELATION STABILITY ACROSS FOLDS
# ============================================================

correlation_stability = (
    feature_correlations.groupby(
        ["panel", "price_area", "feature"],
        as_index=False,
    )
    .agg(
        mean_pearson=("pearson", "mean"),
        std_pearson=("pearson", "std"),
        mean_spearman=("spearman", "mean"),
        std_spearman=("spearman", "std"),
        min_spearman=("spearman", "min"),
        max_spearman=("spearman", "max"),
        folds=("fold_id", "nunique"),
    )
)

correlation_stability["sign_stable"] = (
    np.sign(correlation_stability["min_spearman"])
    == np.sign(correlation_stability["max_spearman"])
)

correlation_stability.to_csv(
    AUDIT_DIR / "notebook_02_correlation_stability.csv",
    index=False,
)

display(
    correlation_stability.sort_values(
        "mean_spearman",
        key=lambda values: values.abs(),
        ascending=False,
    ).head(50)
)

## Feature engineering

The default feature engineering is intentionally limited and interpretable:

- local calendar cycles;
- valid lagged wind production;
- wind-speed powers;
- direction sine/cosine conversion when raw degrees are present;
- provider disagreement and dispersion;
- cross-provider mean and median weather signals.

All lagged target features are built by timestamp alignment. They are not backfilled across missing periods.

In [ ]:
# ============================================================
# 13. LEAKAGE-SAFE FEATURE ENGINEERING
# ============================================================

def add_forecast_origin(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    local_timestamp = (
        result["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )

    delivery_date_string = (
        local_timestamp.dt.strftime("%Y-%m-%d")
    )

    origin_naive = (
        pd.to_datetime(delivery_date_string)
        - pd.Timedelta(days=1)
        + pd.Timedelta(hours=FORECAST_ORIGIN_HOUR_LOCAL)
    )

    origin_local = origin_naive.dt.tz_localize(
        LOCAL_TIMEZONE,
        ambiguous="raise",
        nonexistent="shift_forward",
    )

    result["forecast_origin_utc"] = (
        origin_local.dt.tz_convert("UTC")
    )

    result["delivery_date_local"] = (
        local_timestamp.dt.date.astype(str)
    )

    return result


def add_calendar_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    local_timestamp = (
        result["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )

    result["hour_local"] = local_timestamp.dt.hour
    result["day_of_week"] = local_timestamp.dt.dayofweek
    result["month"] = local_timestamp.dt.month
    result["day_of_year"] = local_timestamp.dt.dayofyear
    result["is_weekend"] = (
        local_timestamp.dt.dayofweek >= 5
    ).astype(int)

    result["hour_sin"] = np.sin(
        2 * np.pi * result["hour_local"] / 24
    )
    result["hour_cos"] = np.cos(
        2 * np.pi * result["hour_local"] / 24
    )
    result["day_of_year_sin"] = np.sin(
        2 * np.pi * result["day_of_year"] / 365.25
    )
    result["day_of_year_cos"] = np.cos(
        2 * np.pi * result["day_of_year"] / 365.25
    )

    return result


def add_information_safe_target_lags(
    frame: pd.DataFrame,
    lag_hours: Sequence[int] = (24, 48, 168),
) -> pd.DataFrame:
    result = frame.copy()

    if "forecast_origin_utc" not in result.columns:
        raise KeyError(
            "forecast_origin_utc must be created before target lags."
        )

    target_lookup = result[
        ["timestamp_utc", "price_area", TARGET_COLUMN]
    ].copy()

    availability_cutoff = (
        result["forecast_origin_utc"]
        - pd.Timedelta(hours=ACTUAL_DATA_LATENCY_HOURS)
    )

    for lag_value in lag_hours:
        lag_column = f"{TARGET_COLUMN}_lag_{lag_value}h"
        source_timestamp_column = f"_source_timestamp_{lag_value}h"

        result[source_timestamp_column] = (
            result["timestamp_utc"]
            - pd.Timedelta(hours=lag_value)
        )

        lookup = target_lookup.rename(
            columns={
                "timestamp_utc": source_timestamp_column,
                TARGET_COLUMN: lag_column,
            }
        )

        result = result.merge(
            lookup,
            on=[source_timestamp_column, "price_area"],
            how="left",
            validate="many_to_one",
        )

        unavailable = (
            result[source_timestamp_column]
            > availability_cutoff
        )

        result.loc[unavailable, lag_column] = np.nan
        result = result.drop(columns=[source_timestamp_column])

    return result


def add_provider_disagreement_features(
    frame: pd.DataFrame,
    providers: Sequence[str],
) -> pd.DataFrame:
    result = frame.copy()

    for region in ("all", "land", "offshore"):
        candidates = []

        for provider in providers:
            preferred = (
                f"{provider}__wind_speed_primary_{region}"
            )

            fallback_100m = (
                f"{provider}__wind_speed_100m_{region}"
            )

            fallback_10m = (
                f"{provider}__wind_speed_10m_{region}"
            )

            for candidate in (
                preferred,
                fallback_100m,
                fallback_10m,
            ):
                if candidate in result.columns:
                    candidates.append(candidate)
                    break

        if len(candidates) < 2:
            continue

        block = result[candidates]

        result[f"provider_mean_wind_{region}"] = block.mean(axis=1)
        result[f"provider_median_wind_{region}"] = block.median(axis=1)
        result[f"provider_std_wind_{region}"] = block.std(axis=1)
        result[f"provider_range_wind_{region}"] = (
            block.max(axis=1) - block.min(axis=1)
        )

    return result


def engineer_features(
    frame: pd.DataFrame,
    providers: Sequence[str],
) -> pd.DataFrame:
    engineered = add_forecast_origin(frame)
    engineered = add_calendar_features(engineered)
    engineered = add_information_safe_target_lags(engineered)
    engineered = add_provider_disagreement_features(
        engineered,
        providers,
    )

    return engineered


d2_engineered = engineer_features(
    d2_panel,
    D2_SOURCES,
)

exact_engineered = engineer_features(
    exact_panel,
    EXACT_SOURCES,
)

for panel_name, frame in (
    ("d2", d2_engineered),
    ("exact", exact_engineered),
):
    invalid_origin_count = int(
        (
            frame["forecast_origin_utc"]
            >= frame["timestamp_utc"]
        ).sum()
    )

    if invalid_origin_count:
        raise RuntimeError(
            f"{panel_name} has {invalid_origin_count} "
            "forecast origins at or after delivery."
        )

print("D2 engineered shape:", d2_engineered.shape)
print("Exact engineered shape:", exact_engineered.shape)

## Baselines

Notebook 02 reports simple, operationally meaningful baselines before fitting LightGBM:

- 24-hour persistence;
- 48-hour persistence;
- 168-hour seasonal persistence;
- expanding historical mean by area and local hour;
- Energinet public forecast when available.

Baselines are evaluated on the same test rows as the machine-learning models.

In [ ]:
# ============================================================
# 14. NUMERIC-SAFE METRICS AND BASELINE UTILITIES
# ============================================================

def _numeric_series(values: pd.Series | np.ndarray | list) -> pd.Series:
    """Return a float64 Series; invalid text becomes NaN."""
    series = pd.Series(values, copy=False)

    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").astype("float64")

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(cleaned, errors="coerce").astype("float64")


def safe_regression_metrics(
    actual: pd.Series,
    predicted: pd.Series,
) -> dict[str, float]:
    actual_numeric = _numeric_series(actual).reset_index(drop=True)
    predicted_numeric = _numeric_series(predicted).reset_index(drop=True)

    valid = pd.DataFrame(
        {
            "actual": actual_numeric,
            "predicted": predicted_numeric,
        }
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if valid.empty:
        return {
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "nmae_mean": np.nan,
            "observations": 0,
        }

    actual_values = valid["actual"].to_numpy(dtype=float)
    predicted_values = valid["predicted"].to_numpy(dtype=float)

    error = predicted_values - actual_values
    mean_actual = float(np.mean(np.abs(actual_values)))

    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "bias": float(np.mean(error)),
        "nmae_mean": float(
            np.mean(np.abs(error)) / mean_actual
            if mean_actual > 0
            else np.nan
        ),
        "observations": int(len(valid)),
    }


def expanding_hourly_climatology(
    train: pd.DataFrame,
    evaluation: pd.DataFrame,
) -> pd.Series:
    """Leakage-safe climatology with hierarchical short-history fallbacks.

    Priority:
      1. price area × month × local hour
      2. price area × local hour
      3. price-area mean
      4. global training mean
    """
    train_local = train["timestamp_utc"].dt.tz_convert(LOCAL_TIMEZONE)
    eval_local = evaluation["timestamp_utc"].dt.tz_convert(LOCAL_TIMEZONE)

    training = train.assign(
        hour_local=train_local.dt.hour,
        month=train_local.dt.month,
    )

    primary_lookup = (
        training.groupby(
            ["price_area", "month", "hour_local"]
        )[TARGET_COLUMN]
        .mean()
    )

    hourly_lookup = (
        training.groupby(
            ["price_area", "hour_local"]
        )[TARGET_COLUMN]
        .mean()
    )

    area_lookup = (
        training.groupby("price_area")[TARGET_COLUMN]
        .mean()
    )

    global_mean = float(training[TARGET_COLUMN].mean())

    primary_keys = pd.MultiIndex.from_arrays(
        [
            evaluation["price_area"].astype(str),
            eval_local.dt.month,
            eval_local.dt.hour,
        ]
    )

    hourly_keys = pd.MultiIndex.from_arrays(
        [
            evaluation["price_area"].astype(str),
            eval_local.dt.hour,
        ]
    )

    prediction = primary_lookup.reindex(
        primary_keys
    ).to_numpy(dtype=float)

    hourly_fallback = hourly_lookup.reindex(
        hourly_keys
    ).to_numpy(dtype=float)

    area_fallback = (
        evaluation["price_area"]
        .astype(str)
        .map(area_lookup)
        .to_numpy(dtype=float)
    )

    prediction = np.where(
        np.isnan(prediction),
        hourly_fallback,
        prediction,
    )
    prediction = np.where(
        np.isnan(prediction),
        area_fallback,
        prediction,
    )
    prediction = np.where(
        np.isnan(prediction),
        global_mean,
        prediction,
    )

    return pd.Series(
        prediction,
        index=evaluation.index,
        name="climatology_prediction",
        dtype="float64",
    )


In [ ]:
# ============================================================
# 14A. VERIFY THAT THE NUMERIC-SAFE METRIC FUNCTION IS ACTIVE
# ============================================================

import inspect

metric_source = inspect.getsource(safe_regression_metrics)

assert "_numeric_series" in metric_source
assert "to_numpy(dtype=float)" in metric_source

print("Numeric-safe metric function is active.")
print(
    safe_regression_metrics(
        pd.Series([1.0, "2.0", "3,0"]),
        pd.Series(["1.5", 2.5, "4,0"]),
    )
)

In [ ]:
# ============================================================
# 15. GENERATE BASELINE PREDICTIONS FOR ALL FOLDS
# ============================================================

def baseline_predictions_for_panel(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    *,
    panel_name: str,
) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []

    lag_columns = {
        "persistence_24h": f"{TARGET_COLUMN}_lag_24h",
        "persistence_48h": f"{TARGET_COLUMN}_lag_48h",
        "persistence_168h": f"{TARGET_COLUMN}_lag_168h",
    }

    benchmark_candidates = [
        column
        for column in frame.columns
        if matches_any(column.lower(), BENCHMARK_PATTERNS)
    ]

    for fold in folds:
        masks = fold_masks(frame, fold)
        train = frame.loc[masks["train"]].copy()
        test = frame.loc[masks["test"]].copy()

        climatology = expanding_hourly_climatology(
            train,
            test,
        )

        prediction_map: dict[str, pd.Series] = {
            model_name: pd.to_numeric(
                test[column],
                errors="coerce",
            )
            for model_name, column in lag_columns.items()
            if column in test.columns
        }

        prediction_map["hourly_climatology"] = climatology

        for benchmark_column in benchmark_candidates:
            benchmark_numeric = pd.to_numeric(
                test[benchmark_column],
                errors="coerce",
            )

            # Skip columns that contain no usable numeric benchmark values.
            if benchmark_numeric.notna().any():
                prediction_map[
                    f"benchmark::{benchmark_column}"
                ] = benchmark_numeric

        for model_name, predictions in prediction_map.items():
            block = test[
                ["timestamp_utc", "price_area", TARGET_COLUMN]
            ].copy()

            block[TARGET_COLUMN] = pd.to_numeric(
                block[TARGET_COLUMN],
                errors="coerce",
            )

            block["panel"] = panel_name
            block["fold_id"] = fold.fold_id
            block["split"] = "test"
            block["model"] = model_name
            block["prediction"] = pd.to_numeric(
                pd.Series(
                    predictions,
                    index=test.index,
                ),
                errors="coerce",
            ).to_numpy(dtype=float)

            rows.append(block)

    return pd.concat(rows, ignore_index=True)


d2_baseline_predictions = baseline_predictions_for_panel(
    d2_engineered,
    d2_folds,
    panel_name="d2",
)

exact_baseline_predictions = baseline_predictions_for_panel(
    exact_engineered,
    exact_folds,
    panel_name="exact",
)

baseline_predictions = pd.concat(
    [d2_baseline_predictions, exact_baseline_predictions],
    ignore_index=True,
)

print(
    baseline_predictions.groupby(
        ["panel", "model"]
    )["prediction"]
    .agg(
        rows="size",
        numeric_values="count",
    )
)

display(baseline_predictions.head())

In [ ]:
# ============================================================
# 16. BASELINE METRIC TABLE — NUMERIC-SAFE
# ============================================================

# Sanitize once at the table boundary. This also fixes stale objects
# produced by an earlier execution of the baseline-generation cell.
baseline_predictions = baseline_predictions.copy()

baseline_predictions[TARGET_COLUMN] = _numeric_series(
    baseline_predictions[TARGET_COLUMN]
).to_numpy(dtype=float)

baseline_predictions["prediction"] = _numeric_series(
    baseline_predictions["prediction"]
).to_numpy(dtype=float)

print(
    "Metric input dtypes:",
    baseline_predictions[
        [TARGET_COLUMN, "prediction"]
    ].dtypes.to_dict(),
)

baseline_metric_rows = []

for keys, group in baseline_predictions.groupby(
    ["panel", "fold_id", "price_area", "model"]
):
    panel_name, fold_id, price_area, model_name = keys

    baseline_metric_rows.append(
        {
            "panel": panel_name,
            "fold_id": fold_id,
            "price_area": price_area,
            "model": model_name,
            **safe_regression_metrics(
                group[TARGET_COLUMN],
                group["prediction"],
            ),
        }
    )

baseline_metrics = pd.DataFrame(baseline_metric_rows)

baseline_metrics.to_csv(
    TABLE_DIR / "notebook_02_baseline_metrics.csv",
    index=False,
)

display(
    baseline_metrics.sort_values(
        ["panel", "price_area", "mae"]
    )
)

## Provider-specific LightGBM experts

Each provider expert receives:

- that provider's weather features;
- shared calendar and valid lag features;
- no other provider's weather fields;
- the same base model family and comparable tuning budget.

Hyperparameters may be tuned inside each training-validation fold, but no test observations may influence the selection.

In [ ]:
# ============================================================
# 17. MODELLING DEPENDENCIES
# ============================================================

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor

    LIGHTGBM_AVAILABLE = True
    print("LightGBM version:", lgb.__version__)
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print(
        "LightGBM is not installed. Install before FULL mode with: "
        "py -m pip install lightgbm"
    )

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import mutual_info_regression
from scipy.optimize import minimize

In [ ]:
# ============================================================
# 18. PROVIDER AND SHARED FEATURE SETS
# ============================================================

ENGINEERED_SHARED_FEATURES = (
    "hour_local",
    "day_of_week",
    "month",
    "day_of_year",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    f"{TARGET_COLUMN}_lag_24h",
    f"{TARGET_COLUMN}_lag_48h",
    f"{TARGET_COLUMN}_lag_168h",
)


def provider_feature_set(
    frame: pd.DataFrame,
    provider: str,
    registry: pd.DataFrame,
) -> list[str]:
    allowed_original = set(
        registry.loc[
            registry["decision_time_status"].eq("allowed"),
            "feature",
        ]
    )

    provider_columns = [
        column
        for column in frame.columns
        if (
            provider.lower() in column.lower()
            and pd.api.types.is_numeric_dtype(frame[column])
            and column in allowed_original
        )
    ]

    shared_columns = [
        column
        for column in ENGINEERED_SHARED_FEATURES
        if (
            column in frame.columns
            and pd.api.types.is_numeric_dtype(frame[column])
        )
    ]

    return sorted(set(provider_columns + shared_columns))


def pooled_feature_set(
    frame: pd.DataFrame,
    registry: pd.DataFrame,
) -> list[str]:
    allowed_original = set(
        registry.loc[
            registry["decision_time_status"].eq("allowed"),
            "feature",
        ]
    )

    engineered_columns = [
        column
        for column in frame.columns
        if (
            column in ENGINEERED_SHARED_FEATURES
            or column.startswith("provider_")
        )
    ]

    candidates = sorted(
        allowed_original.union(engineered_columns)
    )

    return [
        column
        for column in candidates
        if (
            column in frame.columns
            and pd.api.types.is_numeric_dtype(frame[column])
            and column != TARGET_COLUMN
        )
    ]


feature_set_summary = []

for panel_name, frame, providers, registry in (
    ("d2", d2_engineered, D2_SOURCES, d2_feature_registry),
    (
        "exact",
        exact_engineered,
        EXACT_SOURCES,
        exact_feature_registry,
    ),
):
    for provider in providers:
        features = provider_feature_set(
            frame,
            provider,
            registry,
        )

        feature_set_summary.append(
            {
                "panel": panel_name,
                "model": provider,
                "feature_count": len(features),
            }
        )

    feature_set_summary.append(
        {
            "panel": panel_name,
            "model": "pooled",
            "feature_count": len(
                pooled_feature_set(frame, registry)
            ),
        }
    )

feature_set_summary = pd.DataFrame(feature_set_summary)
display(feature_set_summary)

if (feature_set_summary["feature_count"] == 0).any():
    raise RuntimeError(
        "At least one expert has an empty feature set."
    )

In [ ]:
# ============================================================
# 19. INNER-SELECTION AND REFIT UTILITIES
# ============================================================

@dataclass
class SelectedLightGBM:
    candidate_name: str
    parameters: dict[str, Any]
    best_iteration: int
    inner_mae: float


def last_n_local_days_mask(
    frame: pd.DataFrame,
    days: int,
) -> pd.Series:
    dates = local_dates(frame)
    unique_dates = pd.DatetimeIndex(
        dates.drop_duplicates().sort_values()
    )

    if len(unique_dates) <= days + 14:
        split_position = max(1, int(len(unique_dates) * 0.8))
        validation_dates = unique_dates[split_position:]
    else:
        validation_dates = unique_dates[-days:]

    return dates.isin(validation_dates)


def select_lightgbm_configuration(
    outer_train: pd.DataFrame,
    features: Sequence[str],
    *,
    panel_name: str,
    seed: int,
) -> SelectedLightGBM:
    if not LIGHTGBM_AVAILABLE:
        raise RuntimeError(
            "LightGBM is unavailable. Install it before FULL mode."
        )

    inner_days = (
        INNER_VALIDATION_DAYS_D2
        if panel_name == "d2"
        else INNER_VALIDATION_DAYS_EXACT
    )

    inner_validation_mask = last_n_local_days_mask(
        outer_train,
        inner_days,
    )

    inner_train = outer_train.loc[
        ~inner_validation_mask
    ].copy()

    inner_validation = outer_train.loc[
        inner_validation_mask
    ].copy()

    if inner_train.empty or inner_validation.empty:
        raise RuntimeError(
            f"Inner split is empty for {panel_name}."
        )

    candidates: list[SelectedLightGBM] = []

    for candidate_number, candidate in enumerate(
        LIGHTGBM_CANDIDATES,
        start=1,
    ):
        params = {
            **LIGHTGBM_BASE_PARAMS,
            **{
                key: value
                for key, value in candidate.items()
                if key != "name"
            },
            "n_estimators": MAX_BOOSTING_ROUNDS,
            "random_state": seed + candidate_number,
        }

        model = LGBMRegressor(**params)

        model.fit(
            inner_train[list(features)],
            inner_train[TARGET_COLUMN],
            eval_set=[
                (
                    inner_validation[list(features)],
                    inner_validation[TARGET_COLUMN],
                )
            ],
            eval_metric="mae",
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=EARLY_STOPPING_ROUNDS,
                    verbose=False,
                )
            ],
        )

        best_iteration = int(
            model.best_iteration_
            or MAX_BOOSTING_ROUNDS
        )

        prediction = model.predict(
            inner_validation[list(features)],
            num_iteration=best_iteration,
        )

        inner_mae = float(
            mean_absolute_error(
                inner_validation[TARGET_COLUMN],
                prediction,
            )
        )

        final_parameters = {
            **LIGHTGBM_BASE_PARAMS,
            **{
                key: value
                for key, value in candidate.items()
                if key != "name"
            },
        }

        candidates.append(
            SelectedLightGBM(
                candidate_name=str(candidate["name"]),
                parameters=final_parameters,
                best_iteration=best_iteration,
                inner_mae=inner_mae,
            )
        )

    return min(
        candidates,
        key=lambda item: item.inner_mae,
    )


def fit_fixed_lightgbm(
    frame: pd.DataFrame,
    features: Sequence[str],
    selection: SelectedLightGBM,
    *,
    seed: int,
) -> LGBMRegressor:
    params = {
        **selection.parameters,
        "n_estimators": selection.best_iteration,
        "random_state": seed,
    }

    model = LGBMRegressor(**params)

    model.fit(
        frame[list(features)],
        frame[TARGET_COLUMN],
    )

    return model


def expert_predictions_for_fold(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    features: Sequence[str],
    *,
    panel_name: str,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, SelectedLightGBM]:
    selection = select_lightgbm_configuration(
        train,
        features,
        panel_name=panel_name,
        seed=seed,
    )

    validation_model = fit_fixed_lightgbm(
        train,
        features,
        selection,
        seed=seed + 10_000,
    )

    validation_prediction = validation_model.predict(
        validation[list(features)]
    )

    test_training = pd.concat(
        [train, validation],
        ignore_index=True,
    )

    test_model = fit_fixed_lightgbm(
        test_training,
        features,
        selection,
        seed=seed + 20_000,
    )

    test_prediction = test_model.predict(
        test[list(features)]
    )

    return (
        np.asarray(validation_prediction, dtype=float),
        np.asarray(test_prediction, dtype=float),
        selection,
    )

In [ ]:
# ============================================================
# 20. PROVIDER-EXPERT TRAINING LOOP
# ============================================================

def stable_seed(*parts: Any) -> int:
    payload = "|".join(
        str(part)
        for part in parts
    ).encode("utf-8")

    digest = hashlib.sha256(payload).hexdigest()
    return int(digest[:8], 16)


def train_provider_experts(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    providers: Sequence[str],
    registry: pd.DataFrame,
    *,
    panel_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prediction_blocks: list[pd.DataFrame] = []
    metadata_rows: list[dict[str, Any]] = []

    for fold in folds:
        masks = fold_masks(frame, fold)

        for price_area in PRICE_AREAS:
            area_mask = frame["price_area"].eq(price_area)

            train = frame.loc[
                masks["train"] & area_mask
            ].copy()

            validation = frame.loc[
                masks["validation"] & area_mask
            ].copy()

            test = frame.loc[
                masks["test"] & area_mask
            ].copy()

            for provider in providers:
                features = provider_feature_set(
                    frame,
                    provider,
                    registry,
                )

                seed = stable_seed(
                    RANDOM_SEED,
                    panel_name,
                    fold.fold_id,
                    price_area,
                    provider,
                )

                (
                    validation_prediction,
                    test_prediction,
                    selection,
                ) = expert_predictions_for_fold(
                    train,
                    validation,
                    test,
                    features,
                    panel_name=panel_name,
                    seed=seed,
                )

                for split_name, split_frame, predictions in (
                    (
                        "validation",
                        validation,
                        validation_prediction,
                    ),
                    (
                        "test",
                        test,
                        test_prediction,
                    ),
                ):
                    block = split_frame[
                        [
                            "timestamp_utc",
                            "forecast_origin_utc",
                            "delivery_date_local",
                            "price_area",
                            TARGET_COLUMN,
                        ]
                    ].copy()

                    block["panel"] = panel_name
                    block["fold_id"] = fold.fold_id
                    block["split"] = split_name
                    block["model"] = provider
                    block["prediction"] = predictions

                    prediction_blocks.append(block)

                metadata_rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold.fold_id,
                        "price_area": price_area,
                        "provider": provider,
                        "feature_count": len(features),
                        "candidate_name": selection.candidate_name,
                        "best_iteration": selection.best_iteration,
                        "inner_mae": selection.inner_mae,
                    }
                )

    return (
        pd.concat(prediction_blocks, ignore_index=True),
        pd.DataFrame(metadata_rows),
    )


if RUN_MODE == "FULL":
    (
        d2_expert_predictions,
        d2_expert_metadata,
    ) = train_provider_experts(
        d2_engineered,
        d2_folds,
        D2_SOURCES,
        d2_feature_registry,
        panel_name="d2",
    )

    (
        exact_expert_predictions,
        exact_expert_metadata,
    ) = train_provider_experts(
        exact_engineered,
        exact_folds,
        EXACT_SOURCES,
        exact_feature_registry,
        panel_name="exact",
    )

    expert_predictions = pd.concat(
        [
            d2_expert_predictions,
            exact_expert_predictions,
        ],
        ignore_index=True,
    )

    expert_metadata = pd.concat(
        [
            d2_expert_metadata,
            exact_expert_metadata,
        ],
        ignore_index=True,
    )

    expert_predictions.to_parquet(
        PREDICTION_DIR
        / "notebook_02_expert_validation_test_predictions.parquet",
        index=False,
    )

    expert_metadata.to_csv(
        TABLE_DIR / "notebook_02_expert_training_metadata.csv",
        index=False,
    )
else:
    expert_predictions = pd.DataFrame()
    expert_metadata = pd.DataFrame()

    print(
        "AUDIT mode: provider-expert training was skipped. "
        "Both data tracks were still loaded and audited."
    )

## Ensemble methods

The notebook evaluates:

- equal weighting;
- inverse validation-MAE weighting;
- constrained MAE weighting;
- constrained MSE weighting;
- best validation expert;
- linear stacking;
- dynamic exponentially weighted error weighting;
- pooled LightGBM.

All static weights are estimated from validation predictions only.

In [ ]:
# ============================================================
# 21. EXPERT-PREDICTION MATRICES
# ============================================================

def prediction_matrix(
    predictions: pd.DataFrame,
    *,
    split: str,
) -> pd.DataFrame:
    selected = predictions.loc[
        predictions["split"].eq(split)
    ].copy()

    wide = (
        selected.pivot_table(
            index=[
                "panel",
                "fold_id",
                "timestamp_utc",
                "forecast_origin_utc",
                "delivery_date_local",
                "price_area",
                TARGET_COLUMN,
            ],
            columns="model",
            values="prediction",
            aggfunc="first",
        )
        .reset_index()
    )

    wide.columns.name = None
    return wide


if not expert_predictions.empty:
    validation_matrix = prediction_matrix(
        expert_predictions,
        split="validation",
    )

    test_matrix = prediction_matrix(
        expert_predictions,
        split="test",
    )

    display(validation_matrix.head())
else:
    validation_matrix = pd.DataFrame()
    test_matrix = pd.DataFrame()

In [ ]:
# ============================================================
# 22. VERIFIED STATIC ENSEMBLE-WEIGHT ALGORITHMS
# ============================================================

@dataclass
class WeightFit:
    method: str
    weights: np.ndarray
    validation_objective: float
    success: bool
    message: str
    intercept: float = 0.0
    diagnostics: dict[str, Any] | None = None


def normalize_simplex(
    weights: np.ndarray,
) -> np.ndarray:
    weights = np.asarray(weights, dtype=float)
    weights = np.clip(weights, 0.0, None)

    total = float(weights.sum())

    if not np.isfinite(total) or total <= 0:
        return np.full(
            len(weights),
            1.0 / len(weights),
        )

    return weights / total


def validation_mae_by_expert(
    X: np.ndarray,
    y: np.ndarray,
) -> np.ndarray:
    return np.mean(
        np.abs(X - y[:, None]),
        axis=0,
    )


def fit_equal_weights(
    X: np.ndarray,
    y: np.ndarray,
) -> WeightFit:
    weights = np.full(
        X.shape[1],
        1.0 / X.shape[1],
    )

    objective = float(
        np.mean(np.abs(y - X @ weights))
    )

    return WeightFit(
        method="equal_weight",
        weights=weights,
        validation_objective=objective,
        success=True,
        message="closed_form",
    )


def fit_best_validation_expert(
    X: np.ndarray,
    y: np.ndarray,
) -> WeightFit:
    expert_mae = validation_mae_by_expert(X, y)
    best_position = int(np.nanargmin(expert_mae))

    weights = np.zeros(X.shape[1], dtype=float)
    weights[best_position] = 1.0

    return WeightFit(
        method="best_validation_expert",
        weights=weights,
        validation_objective=float(
            expert_mae[best_position]
        ),
        success=True,
        message="one_hot_lowest_validation_mae",
        diagnostics={
            "best_position": best_position,
            "expert_mae": expert_mae.tolist(),
        },
    )


def fit_inverse_mae_weights(
    X: np.ndarray,
    y: np.ndarray,
    epsilon: float = 1e-8,
) -> WeightFit:
    expert_mae = validation_mae_by_expert(X, y)

    weights = normalize_simplex(
        1.0 / (expert_mae + epsilon)
    )

    objective = float(
        np.mean(np.abs(y - X @ weights))
    )

    return WeightFit(
        method="inverse_mae",
        weights=weights,
        validation_objective=objective,
        success=True,
        message="inverse_validation_mae",
        diagnostics={
            "expert_mae": expert_mae.tolist(),
        },
    )


def fit_dirichlet_search(
    X: np.ndarray,
    y: np.ndarray,
    seed: int,
) -> WeightFit:
    rng = np.random.default_rng(seed)
    n_experts = X.shape[1]

    baseline_fits = [
        fit_equal_weights(X, y),
        fit_best_validation_expert(X, y),
        fit_inverse_mae_weights(X, y),
    ]

    best_fit = min(
        baseline_fits,
        key=lambda fit: fit.validation_objective,
    )

    best_weights = best_fit.weights.copy()
    best_objective = float(
        best_fit.validation_objective
    )

    draws_per_alpha = max(
        1,
        DIRICHLET_DRAWS // len(DIRICHLET_ALPHAS),
    )

    evaluated = 0

    for alpha in DIRICHLET_ALPHAS:
        remaining = draws_per_alpha

        while remaining > 0:
            batch_size = min(
                DIRICHLET_BATCH_SIZE,
                remaining,
            )

            candidates = rng.dirichlet(
                np.full(n_experts, alpha),
                size=batch_size,
            )

            predictions = X @ candidates.T

            objectives = np.mean(
                np.abs(
                    y[:, None] - predictions
                ),
                axis=0,
            )

            position = int(np.argmin(objectives))
            candidate_objective = float(
                objectives[position]
            )

            if candidate_objective < best_objective:
                best_objective = candidate_objective
                best_weights = candidates[position].copy()

            evaluated += batch_size
            remaining -= batch_size

    return WeightFit(
        method="dirichlet_mae",
        weights=normalize_simplex(best_weights),
        validation_objective=best_objective,
        success=True,
        message="random_search_complete",
        diagnostics={
            "draws_evaluated": evaluated,
            "alphas": list(DIRICHLET_ALPHAS),
        },
    )


def fit_slsqp_weights(
    X: np.ndarray,
    y: np.ndarray,
    objective_name: str,
    seed: int,
    fallback: WeightFit,
) -> WeightFit:
    if objective_name not in {"mae", "mse"}:
        raise ValueError(
            "objective_name must be mae or mse."
        )

    def objective(weights: np.ndarray) -> float:
        residual = y - X @ weights

        if objective_name == "mae":
            return float(np.mean(np.abs(residual)))

        return float(np.mean(residual ** 2))

    rng = np.random.default_rng(seed)
    n_experts = X.shape[1]

    starts = [
        fit_equal_weights(X, y).weights,
        fit_inverse_mae_weights(X, y).weights,
        fit_best_validation_expert(X, y).weights,
        fallback.weights,
    ]

    starts.extend(
        rng.dirichlet(np.ones(n_experts))
        for _ in range(SLSQP_RANDOM_STARTS)
    )

    constraint = {
        "type": "eq",
        "fun": lambda weights: (
            np.sum(weights) - 1.0
        ),
    }

    bounds = [
        (0.0, 1.0)
        for _ in range(n_experts)
    ]

    attempts = []
    best_result = None

    for start_number, start in enumerate(
        starts,
        start=1,
    ):
        result = minimize(
            objective,
            x0=normalize_simplex(start),
            method="SLSQP",
            bounds=bounds,
            constraints=[constraint],
            options={
                "maxiter": 2_000,
                "ftol": 1e-10,
                "disp": False,
            },
        )

        candidate_weights = normalize_simplex(
            result.x
        )

        candidate_objective = objective(
            candidate_weights
        )

        attempts.append(
            {
                "start": start_number,
                "success": bool(result.success),
                "objective": candidate_objective,
                "message": str(result.message),
            }
        )

        if (
            np.isfinite(candidate_objective)
            and (
                best_result is None
                or candidate_objective < best_result[0]
            )
        ):
            best_result = (
                candidate_objective,
                candidate_weights,
                result,
            )

    method_name = (
        "slsqp_mae"
        if objective_name == "mae"
        else "slsqp_mse"
    )

    if best_result is None:
        return WeightFit(
            method=method_name,
            weights=fallback.weights.copy(),
            validation_objective=float(
                fallback.validation_objective
            ),
            success=False,
            message=(
                "all_slsqp_attempts_failed_fallback_used"
            ),
            diagnostics={
                "attempts": attempts,
                "fallback_method": fallback.method,
            },
        )

    optimized_value, weights, result = best_result

    validation_mae = float(
        np.mean(np.abs(y - X @ weights))
    )

    return WeightFit(
        method=method_name,
        weights=weights,
        validation_objective=validation_mae,
        success=bool(result.success),
        message=str(result.message),
        diagnostics={
            "optimized_objective": objective_name,
            "optimized_objective_value": float(
                optimized_value
            ),
            "attempts": attempts,
        },
    )


def add_median_intercept(
    base_fit: WeightFit,
    X: np.ndarray,
    y: np.ndarray,
) -> WeightFit:
    validation_prediction = (
        X @ base_fit.weights
    )

    intercept = float(
        np.median(y - validation_prediction)
    )

    corrected_prediction = np.clip(
        validation_prediction + intercept,
        0.0,
        None,
    )

    validation_mae = float(
        np.mean(
            np.abs(y - corrected_prediction)
        )
    )

    return WeightFit(
        method="slsqp_mae_median_intercept",
        weights=base_fit.weights.copy(),
        validation_objective=validation_mae,
        success=base_fit.success,
        message=(
            "secondary_median_intercept_robustness"
        ),
        intercept=intercept,
        diagnostics={
            "base_method": base_fit.method,
        },
    )


def fit_static_methods(
    X: np.ndarray,
    y: np.ndarray,
    *,
    seed: int,
) -> list[WeightFit]:
    dirichlet = fit_dirichlet_search(
        X,
        y,
        seed=seed,
    )

    slsqp_mae = fit_slsqp_weights(
        X,
        y,
        objective_name="mae",
        seed=seed + 1,
        fallback=dirichlet,
    )

    slsqp_mse = fit_slsqp_weights(
        X,
        y,
        objective_name="mse",
        seed=seed + 2,
        fallback=dirichlet,
    )

    return [
        fit_equal_weights(X, y),
        fit_best_validation_expert(X, y),
        fit_inverse_mae_weights(X, y),
        dirichlet,
        slsqp_mae,
        slsqp_mse,
        add_median_intercept(
            slsqp_mae,
            X,
            y,
        ),
    ]

In [ ]:
# ============================================================
# 23. FIT STATIC WEIGHTS ON OUTER VALIDATION ONLY
# ============================================================

def fit_static_weights_by_group(
    validation_wide: pd.DataFrame,
    panel_provider_map: dict[str, Sequence[str]],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for keys, group in validation_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        experts = [
            expert
            for expert in panel_provider_map[panel_name]
            if expert in group.columns
        ]

        complete = group[
            [TARGET_COLUMN, *experts]
        ].dropna()

        if complete.empty:
            raise RuntimeError(
                f"No complete validation rows for "
                f"{panel_name}/{fold_id}/{price_area}."
            )

        X = complete[experts].to_numpy(dtype=float)
        y = complete[TARGET_COLUMN].to_numpy(dtype=float)

        seed = stable_seed(
            RANDOM_SEED,
            panel_name,
            fold_id,
            price_area,
            "static_weighting",
        )

        for fit in fit_static_methods(
            X,
            y,
            seed=seed,
        ):
            for expert, weight in zip(
                experts,
                fit.weights,
            ):
                rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold_id,
                        "price_area": price_area,
                        "ensemble_method": fit.method,
                        "expert": expert,
                        "weight": float(weight),
                        "intercept": float(fit.intercept),
                        "validation_mae": float(
                            fit.validation_objective
                        ),
                        "success": bool(fit.success),
                        "message": fit.message,
                        "diagnostics_json": json.dumps(
                            fit.diagnostics,
                            default=str,
                        ),
                    }
                )

    return pd.DataFrame(rows)


PANEL_PROVIDER_MAP = {
    "d2": D2_SOURCES,
    "exact": EXACT_SOURCES,
}

if not validation_matrix.empty:
    ensemble_weights = fit_static_weights_by_group(
        validation_matrix,
        PANEL_PROVIDER_MAP,
    )

    ensemble_weights.to_csv(
        TABLE_DIR / "notebook_02_static_ensemble_weights.csv",
        index=False,
    )

    display(
        ensemble_weights.groupby(
            [
                "panel",
                "price_area",
                "ensemble_method",
            ],
            as_index=False,
        )["validation_mae"].first()
    )
else:
    ensemble_weights = pd.DataFrame()

In [ ]:
# ============================================================
# 24. APPLY FROZEN STATIC WEIGHTS TO TEST
# ============================================================

def apply_static_ensembles(
    test_wide: pd.DataFrame,
    weights: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []

    for keys, group in test_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        group_weights = weights.loc[
            weights["panel"].eq(panel_name)
            & weights["fold_id"].eq(fold_id)
            & weights["price_area"].eq(price_area)
        ]

        for method, method_weights in group_weights.groupby(
            "ensemble_method"
        ):
            experts = method_weights["expert"].tolist()

            weight_vector = (
                method_weights.set_index("expert")["weight"]
                .reindex(experts)
                .to_numpy(dtype=float)
            )

            intercept = float(
                method_weights["intercept"].iloc[0]
            )

            complete_mask = group[experts].notna().all(axis=1)
            selected = group.loc[complete_mask].copy()

            prediction = np.clip(
                selected[experts].to_numpy(dtype=float)
                @ weight_vector
                + intercept,
                0.0,
                None,
            )

            block = selected[
                [
                    "panel",
                    "fold_id",
                    "timestamp_utc",
                    "forecast_origin_utc",
                    "delivery_date_local",
                    "price_area",
                    TARGET_COLUMN,
                ]
            ].copy()

            block["split"] = "test"
            block["model"] = f"ensemble::{method}"
            block["prediction"] = prediction

            rows.append(block)

    return pd.concat(rows, ignore_index=True)


if (
    not test_matrix.empty
    and not ensemble_weights.empty
):
    static_ensemble_predictions = apply_static_ensembles(
        test_matrix,
        ensemble_weights,
    )
else:
    static_ensemble_predictions = pd.DataFrame()

In [ ]:
# ============================================================
# 25. NON-NEGATIVE LINEAR STACKING ROBUSTNESS BENCHMARK
# ============================================================

def fit_and_apply_linear_stack(
    validation_wide: pd.DataFrame,
    test_wide: pd.DataFrame,
    providers: Sequence[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prediction_rows: list[pd.DataFrame] = []
    weight_rows: list[dict[str, Any]] = []

    for keys, validation_group in validation_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        experts = [
            provider
            for provider in providers
            if provider in validation_group.columns
        ]

        test_group = test_wide.loc[
            test_wide["panel"].eq(panel_name)
            & test_wide["fold_id"].eq(fold_id)
            & test_wide["price_area"].eq(price_area)
        ].copy()

        validation_complete = validation_group[
            [TARGET_COLUMN, *experts]
        ].dropna()

        test_complete = test_group.dropna(subset=experts)

        stacker = LinearRegression(
            fit_intercept=True,
            positive=True,
        )

        stacker.fit(
            validation_complete[experts],
            validation_complete[TARGET_COLUMN],
        )

        prediction = np.clip(
            stacker.predict(test_complete[experts]),
            0.0,
            None,
        )

        block = test_complete[
            [
                "panel",
                "fold_id",
                "timestamp_utc",
                "forecast_origin_utc",
                "delivery_date_local",
                "price_area",
                TARGET_COLUMN,
            ]
        ].copy()

        block["split"] = "test"
        block["model"] = "ensemble::linear_stack"
        block["prediction"] = prediction
        prediction_rows.append(block)

        for expert, coefficient in zip(
            experts,
            stacker.coef_,
        ):
            weight_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold_id,
                    "price_area": price_area,
                    "ensemble_method": "linear_stack",
                    "expert": expert,
                    "weight": float(coefficient),
                    "intercept": float(
                        stacker.intercept_
                    ),
                }
            )

    return (
        pd.concat(prediction_rows, ignore_index=True),
        pd.DataFrame(weight_rows),
    )


if not validation_matrix.empty:
    stack_prediction_blocks = []
    stack_weight_blocks = []

    for panel_name, providers in PANEL_PROVIDER_MAP.items():
        predictions, weights = fit_and_apply_linear_stack(
            validation_matrix.loc[
                validation_matrix["panel"].eq(panel_name)
            ],
            test_matrix.loc[
                test_matrix["panel"].eq(panel_name)
            ],
            providers,
        )

        stack_prediction_blocks.append(predictions)
        stack_weight_blocks.append(weights)

    stacking_predictions = pd.concat(
        stack_prediction_blocks,
        ignore_index=True,
    )

    stacking_weights = pd.concat(
        stack_weight_blocks,
        ignore_index=True,
    )
else:
    stacking_predictions = pd.DataFrame()
    stacking_weights = pd.DataFrame()

## Forecast-origin-safe dynamic weighting

Hedge is fitted and applied separately for each panel, fold, and price area.

- Hyperparameters are selected using validation MAE.
- Expert losses are scaled and clipped.
- Shrinkage prevents irreversible expert elimination.
- The final validation and test simulation is chronological.
- Before each forecast origin, only outcomes timestamped at or before the availability cutoff may update the weights.
- All hours sharing the same day-ahead origin use the same pre-prediction weight vector.

In [ ]:
# ============================================================
# 26. FORECAST-ORIGIN-SAFE HEDGE
# ============================================================

@dataclass
class HedgeSimulation:
    predictions: np.ndarray
    pre_prediction_weights: pd.DataFrame
    final_weights: np.ndarray
    mae: float
    loss_scale: float


def hedge_update(
    weights: np.ndarray,
    expert_losses: np.ndarray,
    eta: float,
    shrinkage: float,
    loss_scale: float,
) -> np.ndarray:
    scaled_loss = np.clip(
        expert_losses / max(loss_scale, 1e-8),
        0.0,
        HEDGE_LOSS_CLIP,
    )

    log_weight = (
        np.log(
            np.clip(weights, 1e-300, None)
        )
        - eta * scaled_loss
    )

    log_weight -= np.max(log_weight)

    updated = normalize_simplex(
        np.exp(log_weight)
    )

    if shrinkage > 0:
        equal_weight = np.full(
            len(updated),
            1.0 / len(updated),
        )

        updated = (
            (1.0 - shrinkage) * updated
            + shrinkage * equal_weight
        )

    return normalize_simplex(updated)


def simulate_hedge(
    frame: pd.DataFrame,
    experts: list[str],
    eta: float,
    shrinkage: float,
    *,
    initial_weights: np.ndarray | None = None,
    fixed_loss_scale: float | None = None,
) -> HedgeSimulation:
    ordered = frame.sort_values(
        ["forecast_origin_utc", "timestamp_utc"]
    ).reset_index(drop=True)

    X = ordered[experts].to_numpy(dtype=float)
    y = ordered[TARGET_COLUMN].to_numpy(dtype=float)

    if not np.isfinite(X).all():
        raise AssertionError(
            "Hedge matrix contains missing or non-finite values."
        )

    n_rows, n_experts = X.shape

    weights = (
        np.full(n_experts, 1.0 / n_experts)
        if initial_weights is None
        else normalize_simplex(initial_weights)
    )

    absolute_error_matrix = np.abs(
        X - y[:, None]
    )

    if fixed_loss_scale is None:
        loss_scale = float(
            np.median(absolute_error_matrix)
        )
    else:
        loss_scale = float(fixed_loss_scale)

    if not np.isfinite(loss_scale) or loss_scale <= 0:
        loss_scale = float(
            np.mean(np.abs(y))
        )

    predictions = np.full(
        n_rows,
        np.nan,
        dtype=float,
    )

    processed_for_update = np.zeros(
        n_rows,
        dtype=bool,
    )

    path_rows: list[dict[str, Any]] = []

    grouped_positions = (
        ordered.groupby(
            "forecast_origin_utc",
            sort=True,
        ).indices
    )

    for forecast_origin, positions in grouped_positions.items():
        positions = np.asarray(positions, dtype=int)

        cutoff = (
            pd.Timestamp(forecast_origin)
            - pd.Timedelta(
                hours=ACTUAL_DATA_LATENCY_HOURS
            )
        )

        newly_available = (
            ~processed_for_update
            & (
                ordered["timestamp_utc"]
                <= cutoff
            ).to_numpy()
        )

        if newly_available.any():
            expert_losses = (
                absolute_error_matrix[
                    newly_available
                ].mean(axis=0)
            )

            weights = hedge_update(
                weights=weights,
                expert_losses=expert_losses,
                eta=eta,
                shrinkage=shrinkage,
                loss_scale=loss_scale,
            )

            processed_for_update[
                newly_available
            ] = True

        predictions[positions] = (
            X[positions] @ weights
        )

        delivery_date = ordered.loc[
            positions[0],
            "delivery_date_local",
        ]

        for expert, weight in zip(
            experts,
            weights,
        ):
            path_rows.append(
                {
                    "forecast_origin_utc": forecast_origin,
                    "delivery_date_local": delivery_date,
                    "expert": expert,
                    "weight": float(weight),
                }
            )

    mae = float(
        np.mean(np.abs(y - predictions))
    )

    return HedgeSimulation(
        predictions=np.clip(
            predictions,
            0.0,
            None,
        ),
        pre_prediction_weights=pd.DataFrame(
            path_rows
        ),
        final_weights=weights,
        mae=mae,
        loss_scale=loss_scale,
    )


def fit_and_apply_hedge(
    validation_frame: pd.DataFrame,
    test_frame: pd.DataFrame,
    experts: list[str],
) -> tuple[
    np.ndarray,
    dict[str, Any],
    pd.DataFrame,
]:
    validation_complete = validation_frame.dropna(
        subset=[TARGET_COLUMN, *experts]
    ).copy()

    test_complete = test_frame.dropna(
        subset=[TARGET_COLUMN, *experts]
    ).copy()

    candidate_rows = []

    for eta in HEDGE_ETA_GRID:
        for shrinkage in HEDGE_SHRINKAGE_GRID:
            simulation = simulate_hedge(
                frame=validation_complete,
                experts=experts,
                eta=eta,
                shrinkage=shrinkage,
            )

            candidate_rows.append(
                {
                    "eta": eta,
                    "shrinkage": shrinkage,
                    "validation_mae": simulation.mae,
                    "loss_scale": simulation.loss_scale,
                }
            )

    candidate_report = (
        pd.DataFrame(candidate_rows)
        .sort_values(
            [
                "validation_mae",
                "shrinkage",
                "eta",
            ]
        )
        .reset_index(drop=True)
    )

    selected = candidate_report.iloc[0]

    selected_eta = float(selected["eta"])
    selected_shrinkage = float(
        selected["shrinkage"]
    )
    validation_loss_scale = float(
        selected["loss_scale"]
    )

    # Run validation and test as one chronological stream after
    # hyperparameters and loss scale are frozen from validation.
    validation_stream = validation_complete.assign(
        _hedge_split="validation"
    )

    test_stream = test_complete.assign(
        _hedge_split="test"
    )

    combined = pd.concat(
        [validation_stream, test_stream],
        ignore_index=True,
    ).sort_values(
        ["forecast_origin_utc", "timestamp_utc"]
    ).reset_index(drop=True)

    final_simulation = simulate_hedge(
        frame=combined,
        experts=experts,
        eta=selected_eta,
        shrinkage=selected_shrinkage,
        fixed_loss_scale=validation_loss_scale,
    )

    test_mask = combined["_hedge_split"].eq("test")
    test_predictions = (
        final_simulation.predictions[test_mask.to_numpy()]
    )

    first_test_origin = test_complete[
        "forecast_origin_utc"
    ].min()

    test_weight_path = (
        final_simulation.pre_prediction_weights.loc[
            lambda frame: (
                frame["forecast_origin_utc"]
                >= first_test_origin
            )
        ]
        .reset_index(drop=True)
    )

    report = {
        "eta": selected_eta,
        "shrinkage": selected_shrinkage,
        "validation_mae": float(
            selected["validation_mae"]
        ),
        "validation_loss_scale": validation_loss_scale,
        "candidate_report": candidate_report.to_dict(
            orient="records"
        ),
    }

    return (
        test_predictions,
        report,
        test_weight_path,
    )


if not validation_matrix.empty:
    hedge_prediction_blocks = []
    hedge_report_rows = []
    hedge_weight_blocks = []

    for keys, validation_group in validation_matrix.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        test_group = test_matrix.loc[
            test_matrix["panel"].eq(panel_name)
            & test_matrix["fold_id"].eq(fold_id)
            & test_matrix["price_area"].eq(price_area)
        ].copy()

        experts = [
            expert
            for expert in PANEL_PROVIDER_MAP[panel_name]
            if (
                expert in validation_group.columns
                and expert in test_group.columns
            )
        ]

        validation_complete = validation_group.dropna(
            subset=[TARGET_COLUMN, *experts]
        ).copy()

        test_complete = test_group.dropna(
            subset=[TARGET_COLUMN, *experts]
        ).copy()

        (
            prediction,
            report,
            weight_path,
        ) = fit_and_apply_hedge(
            validation_complete,
            test_complete,
            experts,
        )

        block = test_complete[
            [
                "panel",
                "fold_id",
                "timestamp_utc",
                "forecast_origin_utc",
                "delivery_date_local",
                "price_area",
                TARGET_COLUMN,
            ]
        ].copy()

        block["split"] = "test"
        block["model"] = "ensemble::hedge"
        block["prediction"] = prediction
        hedge_prediction_blocks.append(block)

        hedge_report_rows.append(
            {
                "panel": panel_name,
                "fold_id": fold_id,
                "price_area": price_area,
                **{
                    key: value
                    for key, value in report.items()
                    if key != "candidate_report"
                },
                "candidate_report_json": json.dumps(
                    report["candidate_report"]
                ),
            }
        )

        weight_path["panel"] = panel_name
        weight_path["fold_id"] = fold_id
        weight_path["price_area"] = price_area
        hedge_weight_blocks.append(weight_path)

    hedge_prediction_frame = pd.concat(
        hedge_prediction_blocks,
        ignore_index=True,
    )

    hedge_reports = pd.DataFrame(
        hedge_report_rows
    )

    hedge_weight_history = pd.concat(
        hedge_weight_blocks,
        ignore_index=True,
    )

    hedge_reports.to_csv(
        TABLE_DIR / "notebook_02_hedge_selection.csv",
        index=False,
    )

    hedge_weight_history.to_parquet(
        TABLE_DIR / "notebook_02_hedge_weight_history.parquet",
        index=False,
    )
else:
    hedge_prediction_frame = pd.DataFrame()
    hedge_reports = pd.DataFrame()
    hedge_weight_history = pd.DataFrame()

## Forecast-disagreement-regime weighting

This regime is known at forecast origin because it is calculated only from the current expert forecasts.

For each panel, fold, and area:

1. Compute expert-prediction standard deviation on validation rows.
2. Estimate low/medium/high thresholds from validation quantiles.
3. Fit separate constrained MAE weights within each validation regime.
4. Freeze the thresholds and weights.
5. Apply them to test rows according to their forecast disagreement.
6. Fall back to the global constrained-MAE weights when a validation regime is too small.

In [ ]:
# ============================================================
# 26A. FORECAST-DISAGREEMENT-REGIME WEIGHTING
# ============================================================

def assign_disagreement_regime(
    frame: pd.DataFrame,
    experts: Sequence[str],
    thresholds: tuple[float, float],
) -> pd.Series:
    disagreement = frame[list(experts)].std(
        axis=1,
        ddof=0,
    )

    lower, upper = thresholds

    return pd.cut(
        disagreement,
        bins=[-np.inf, lower, upper, np.inf],
        labels=["low", "medium", "high"],
        include_lowest=True,
    ).astype(str)


def fit_and_apply_regime_weights(
    validation_wide: pd.DataFrame,
    test_wide: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    prediction_blocks = []
    weight_rows = []
    threshold_rows = []

    for keys, validation_group in validation_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        test_group = test_wide.loc[
            test_wide["panel"].eq(panel_name)
            & test_wide["fold_id"].eq(fold_id)
            & test_wide["price_area"].eq(price_area)
        ].copy()

        experts = [
            expert
            for expert in PANEL_PROVIDER_MAP[panel_name]
            if (
                expert in validation_group.columns
                and expert in test_group.columns
            )
        ]

        validation_complete = validation_group.dropna(
            subset=[TARGET_COLUMN, *experts]
        ).copy()

        test_complete = test_group.dropna(
            subset=[TARGET_COLUMN, *experts]
        ).copy()

        validation_disagreement = (
            validation_complete[experts]
            .std(axis=1, ddof=0)
        )

        lower, upper = (
            validation_disagreement.quantile(
                list(REGIME_QUANTILES)
            ).to_numpy(dtype=float)
        )

        thresholds = (float(lower), float(upper))

        validation_complete["disagreement_regime"] = (
            assign_disagreement_regime(
                validation_complete,
                experts,
                thresholds,
            )
        )

        test_complete["disagreement_regime"] = (
            assign_disagreement_regime(
                test_complete,
                experts,
                thresholds,
            )
        )

        X_global = validation_complete[
            experts
        ].to_numpy(dtype=float)

        y_global = validation_complete[
            TARGET_COLUMN
        ].to_numpy(dtype=float)

        global_dirichlet = fit_dirichlet_search(
            X_global,
            y_global,
            seed=stable_seed(
                RANDOM_SEED,
                panel_name,
                fold_id,
                price_area,
                "regime_global_dirichlet",
            ),
        )

        global_fit = fit_slsqp_weights(
            X_global,
            y_global,
            objective_name="mae",
            seed=stable_seed(
                RANDOM_SEED,
                panel_name,
                fold_id,
                price_area,
                "regime_global_slsqp",
            ),
            fallback=global_dirichlet,
        )

        threshold_rows.append(
            {
                "panel": panel_name,
                "fold_id": fold_id,
                "price_area": price_area,
                "lower_threshold": lower,
                "upper_threshold": upper,
            }
        )

        for regime in ("low", "medium", "high"):
            validation_regime = validation_complete.loc[
                validation_complete[
                    "disagreement_regime"
                ].eq(regime)
            ]

            if (
                len(validation_regime)
                >= MIN_REGIME_VALIDATION_ROWS
            ):
                X_regime = validation_regime[
                    experts
                ].to_numpy(dtype=float)

                y_regime = validation_regime[
                    TARGET_COLUMN
                ].to_numpy(dtype=float)

                regime_dirichlet = fit_dirichlet_search(
                    X_regime,
                    y_regime,
                    seed=stable_seed(
                        RANDOM_SEED,
                        panel_name,
                        fold_id,
                        price_area,
                        regime,
                        "dirichlet",
                    ),
                )

                regime_fit = fit_slsqp_weights(
                    X_regime,
                    y_regime,
                    objective_name="mae",
                    seed=stable_seed(
                        RANDOM_SEED,
                        panel_name,
                        fold_id,
                        price_area,
                        regime,
                        "slsqp",
                    ),
                    fallback=regime_dirichlet,
                )

                fallback_used = False
            else:
                regime_fit = global_fit
                fallback_used = True

            for expert, weight in zip(
                experts,
                regime_fit.weights,
            ):
                weight_rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold_id,
                        "price_area": price_area,
                        "regime": regime,
                        "expert": expert,
                        "weight": float(weight),
                        "validation_rows": int(
                            len(validation_regime)
                        ),
                        "fallback_to_global": fallback_used,
                    }
                )

            test_regime = test_complete.loc[
                test_complete[
                    "disagreement_regime"
                ].eq(regime)
            ].copy()

            if test_regime.empty:
                continue

            prediction = np.clip(
                test_regime[experts].to_numpy(dtype=float)
                @ regime_fit.weights,
                0.0,
                None,
            )

            block = test_regime[
                [
                    "panel",
                    "fold_id",
                    "timestamp_utc",
                    "forecast_origin_utc",
                    "delivery_date_local",
                    "price_area",
                    TARGET_COLUMN,
                ]
            ].copy()

            block["split"] = "test"
            block["model"] = (
                "ensemble::regime_disagreement_slsqp_mae"
            )
            block["prediction"] = prediction
            block["disagreement_regime"] = regime

            prediction_blocks.append(block)

    return (
        pd.concat(prediction_blocks, ignore_index=True),
        pd.DataFrame(weight_rows),
        pd.DataFrame(threshold_rows),
    )


if not validation_matrix.empty:
    (
        regime_ensemble_predictions,
        regime_ensemble_weights,
        regime_thresholds,
    ) = fit_and_apply_regime_weights(
        validation_matrix,
        test_matrix,
    )

    regime_ensemble_weights.to_csv(
        TABLE_DIR / "notebook_02_regime_ensemble_weights.csv",
        index=False,
    )

    regime_thresholds.to_csv(
        TABLE_DIR / "notebook_02_regime_thresholds.csv",
        index=False,
    )
else:
    regime_ensemble_predictions = pd.DataFrame()
    regime_ensemble_weights = pd.DataFrame()
    regime_thresholds = pd.DataFrame()

## Pooled LightGBM

The pooled model receives the complete allowed weather feature set from all providers plus shared engineered features. It is a benchmark against explicit expert combination, not a replacement for provider-level interpretability.

In [ ]:
# ============================================================
# 27. POOLED ALL-PROVIDER LIGHTGBM
# ============================================================

def train_pooled_model(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    registry: pd.DataFrame,
    *,
    panel_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prediction_blocks: list[pd.DataFrame] = []
    metadata_rows: list[dict[str, Any]] = []

    features = pooled_feature_set(
        frame,
        registry,
    )

    for fold in folds:
        masks = fold_masks(frame, fold)

        for price_area in PRICE_AREAS:
            area_mask = frame["price_area"].eq(price_area)

            train = frame.loc[
                masks["train"] & area_mask
            ].copy()

            validation = frame.loc[
                masks["validation"] & area_mask
            ].copy()

            test = frame.loc[
                masks["test"] & area_mask
            ].copy()

            seed = stable_seed(
                RANDOM_SEED,
                panel_name,
                fold.fold_id,
                price_area,
                "pooled",
            )

            (
                validation_prediction,
                test_prediction,
                selection,
            ) = expert_predictions_for_fold(
                train,
                validation,
                test,
                features,
                panel_name=panel_name,
                seed=seed,
            )

            for split_name, split_frame, prediction in (
                (
                    "validation",
                    validation,
                    validation_prediction,
                ),
                (
                    "test",
                    test,
                    test_prediction,
                ),
            ):
                block = split_frame[
                    [
                        "timestamp_utc",
                        "forecast_origin_utc",
                        "delivery_date_local",
                        "price_area",
                        TARGET_COLUMN,
                    ]
                ].copy()

                block["panel"] = panel_name
                block["fold_id"] = fold.fold_id
                block["split"] = split_name
                block["model"] = "pooled_lightgbm"
                block["prediction"] = prediction

                prediction_blocks.append(block)

            metadata_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold.fold_id,
                    "price_area": price_area,
                    "feature_count": len(features),
                    "candidate_name": selection.candidate_name,
                    "best_iteration": selection.best_iteration,
                    "inner_mae": selection.inner_mae,
                }
            )

    return (
        pd.concat(prediction_blocks, ignore_index=True),
        pd.DataFrame(metadata_rows),
    )


if RUN_MODE == "FULL":
    (
        d2_pooled_predictions,
        d2_pooled_metadata,
    ) = train_pooled_model(
        d2_engineered,
        d2_folds,
        d2_feature_registry,
        panel_name="d2",
    )

    (
        exact_pooled_predictions,
        exact_pooled_metadata,
    ) = train_pooled_model(
        exact_engineered,
        exact_folds,
        exact_feature_registry,
        panel_name="exact",
    )

    pooled_predictions_all_splits = pd.concat(
        [
            d2_pooled_predictions,
            exact_pooled_predictions,
        ],
        ignore_index=True,
    )

    pooled_predictions = (
        pooled_predictions_all_splits.loc[
            pooled_predictions_all_splits[
                "split"
            ].eq("test")
        ].copy()
    )

    pooled_metadata = pd.concat(
        [
            d2_pooled_metadata,
            exact_pooled_metadata,
        ],
        ignore_index=True,
    )

    pooled_predictions_all_splits.to_parquet(
        PREDICTION_DIR
        / "notebook_02_pooled_validation_test_predictions.parquet",
        index=False,
    )

    pooled_metadata.to_csv(
        TABLE_DIR / "notebook_02_pooled_training_metadata.csv",
        index=False,
    )
else:
    pooled_predictions_all_splits = pd.DataFrame()
    pooled_predictions = pd.DataFrame()
    pooled_metadata = pd.DataFrame()

## Out-of-sample evaluation

Primary metrics:

- MAE;
- RMSE;
- bias;
- normalized MAE;
- daily aggregate absolute error;
- ramp error;
- performance by season, hour, wind regime, and provider-disagreement regime;
- fold-to-fold stability;
- expert-weight stability.

The test sets are the only rows used for final comparative reporting.

In [ ]:
# ============================================================
# 28. COMBINE ALL TEST PREDICTIONS
# ============================================================

prediction_frames = [
    baseline_predictions,
]

optional_prediction_frames = [
    (
        expert_predictions.loc[
            expert_predictions["split"].eq("test")
        ]
        if not expert_predictions.empty
        else pd.DataFrame()
    ),
    static_ensemble_predictions,
    stacking_predictions,
    hedge_prediction_frame,
    regime_ensemble_predictions,
    pooled_predictions,
]

for optional_frame in optional_prediction_frames:
    if not optional_frame.empty:
        prediction_frames.append(optional_frame)

all_test_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
    sort=False,
)

all_test_predictions.to_parquet(
    PREDICTION_DIR
    / "notebook_02_test_predictions.parquet",
    index=False,
)

print("Combined prediction rows:", len(all_test_predictions))

display(
    all_test_predictions.groupby(
        ["panel", "model"],
        as_index=False,
    ).size()
)

In [ ]:
# ============================================================
# 29. PRIMARY ACCURACY TABLE
# ============================================================

metric_rows = []

for keys, group in all_test_predictions.groupby(
    ["panel", "fold_id", "price_area", "model"]
):
    panel_name, fold_id, price_area, model_name = keys

    metric_rows.append(
        {
            "panel": panel_name,
            "fold_id": fold_id,
            "price_area": price_area,
            "model": model_name,
            **safe_regression_metrics(
                group[TARGET_COLUMN],
                group["prediction"],
            ),
        }
    )

model_metrics = pd.DataFrame(metric_rows)

model_metrics.to_csv(
    TABLE_DIR / "notebook_02_model_metrics_by_fold.csv",
    index=False,
)

display(
    model_metrics.sort_values(
        ["panel", "price_area", "fold_id", "mae"]
    )
)

In [ ]:
# ============================================================
# 30. AGGREGATE ACCURACY AND STABILITY
# ============================================================

metric_summary = (
    model_metrics.groupby(
        ["panel", "price_area", "model"],
        as_index=False,
    )
    .agg(
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_bias=("bias", "mean"),
        mean_nmae=("nmae_mean", "mean"),
        folds=("fold_id", "nunique"),
        observations=("observations", "sum"),
    )
)

metric_summary["mae_cv"] = (
    metric_summary["std_mae"]
    / metric_summary["mean_mae"]
)

metric_summary.to_csv(
    TABLE_DIR / "notebook_02_model_metric_summary.csv",
    index=False,
)

display(
    metric_summary.sort_values(
        ["panel", "price_area", "mean_mae"]
    )
)

In [ ]:
# ============================================================
# 31. REGIME LABELS
# ============================================================

def add_evaluation_regimes(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    result = predictions.copy()

    local_timestamp = (
        result["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )

    result["hour_local"] = local_timestamp.dt.hour
    result["month"] = local_timestamp.dt.month

    result["season"] = pd.cut(
        result["month"],
        bins=[0, 2, 5, 8, 11, 12],
        labels=[
            "winter",
            "spring",
            "summer",
            "autumn",
            "winter_end",
        ],
        include_lowest=True,
    ).astype(str)

    result.loc[
        result["season"].eq("winter_end"),
        "season",
    ] = "winter"

    result["wind_regime"] = (
        result.groupby(
            ["panel", "price_area"]
        )[TARGET_COLUMN]
        .transform(
            lambda values: pd.qcut(
                values.rank(method="first"),
                q=3,
                labels=["low", "medium", "high"],
            )
        )
    )

    result["absolute_error"] = (
        result["prediction"]
        - result[TARGET_COLUMN]
    ).abs()

    return result


regime_predictions = add_evaluation_regimes(
    all_test_predictions
)

In [ ]:
# ============================================================
# 32. REGIME PERFORMANCE TABLE
# ============================================================

regime_metric_rows = []

for regime_name in (
    "season",
    "hour_local",
    "wind_regime",
):
    for keys, group in regime_predictions.groupby(
        [
            "panel",
            "price_area",
            "model",
            regime_name,
        ],
        dropna=False,
    ):
        panel_name, price_area, model_name, regime_value = keys

        regime_metric_rows.append(
            {
                "panel": panel_name,
                "price_area": price_area,
                "model": model_name,
                "regime_type": regime_name,
                "regime_value": str(regime_value),
                **safe_regression_metrics(
                    group[TARGET_COLUMN],
                    group["prediction"],
                ),
            }
        )

regime_metrics = pd.DataFrame(regime_metric_rows)

regime_metrics.to_csv(
    TABLE_DIR / "notebook_02_regime_metrics.csv",
    index=False,
)

display(regime_metrics.head(50))

In [ ]:
# ============================================================
# 33. RESIDUAL CORRELATION AMONG VALID PROVIDER EXPERTS
# ============================================================

if not expert_predictions.empty:
    expert_test = expert_predictions.loc[
        expert_predictions["split"].eq("test")
    ].copy()

    expert_test["residual"] = (
        expert_test["prediction"]
        - expert_test[TARGET_COLUMN]
    )

    residual_correlation_rows = []

    for keys, group in expert_test.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        providers = [
            provider
            for provider in PANEL_PROVIDER_MAP[panel_name]
            if provider in set(group["model"])
        ]

        residual_wide = group.pivot_table(
            index="timestamp_utc",
            columns="model",
            values="residual",
            aggfunc="first",
        )

        for expert_a, expert_b in combinations(providers, 2):
            pair = residual_wide[
                [expert_a, expert_b]
            ].replace(
                [np.inf, -np.inf],
                np.nan,
            ).dropna()

            correlation = (
                float(pair[expert_a].corr(pair[expert_b]))
                if len(pair) >= 2
                else np.nan
            )

            residual_correlation_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold_id,
                    "price_area": price_area,
                    "expert_a": expert_a,
                    "expert_b": expert_b,
                    "common_observations": int(len(pair)),
                    "residual_correlation": correlation,
                }
            )

    residual_correlations = pd.DataFrame(
        residual_correlation_rows
    )

    residual_correlations.to_csv(
        AUDIT_DIR
        / "notebook_02_expert_residual_correlations.csv",
        index=False,
    )

    display(
        residual_correlations.sort_values(
            ["panel", "price_area", "fold_id", "residual_correlation"]
        ).head(100)
    )
else:
    residual_correlations = pd.DataFrame()
    print("Residual-correlation analysis requires FULL mode.")


In [ ]:
# ============================================================
# 34. ENSEMBLE-WEIGHT STABILITY
# ============================================================

weight_frames = []

if not ensemble_weights.empty:
    weight_frames.append(
        ensemble_weights.assign(
            weight_source="static"
        )
    )

if not stacking_weights.empty:
    weight_frames.append(
        stacking_weights.assign(
            weight_source="stacking"
        )
    )

if weight_frames:
    combined_weights = pd.concat(
        weight_frames,
        ignore_index=True,
        sort=False,
    )

    weight_stability = (
        combined_weights.groupby(
            [
                "panel",
                "price_area",
                "ensemble_method",
                "expert",
            ],
            as_index=False,
        )
        .agg(
            mean_weight=("weight", "mean"),
            std_weight=("weight", "std"),
            min_weight=("weight", "min"),
            max_weight=("weight", "max"),
            folds=("fold_id", "nunique"),
        )
    )

    weight_stability.to_csv(
        TABLE_DIR / "notebook_02_weight_stability.csv",
        index=False,
    )

    display(weight_stability)
else:
    combined_weights = pd.DataFrame()
    weight_stability = pd.DataFrame()
    print("Weight-stability analysis requires FULL mode.")

## Exact-run and D2 comparison

The two tracks are not concatenated. Comparisons are restricted to overlapping timestamps and use separately trained models.

Useful comparisons:

- best D2 expert versus best exact-run expert;
- pooled D2 versus pooled exact-run;
- equal-weight D2 versus equal-weight exact-run;
- coverage and sample-size trade-offs;
- whether exact information-time realism changes provider ranking.

## Common-support evaluation and paired inference

All comparisons below are paired on identical delivery timestamps. A negative
loss difference means `model_a` has lower absolute error than `model_b`.

The formal comparison list is fixed in code. It is a follow-up to the v4
exploratory run, so the resulting p-values should be interpreted as
confirmation within the available sample rather than as evidence from a wholly
independent holdout.


In [ ]:
# ============================================================
# 35. PAIRED COMMON-SUPPORT UTILITIES
# ============================================================

def paired_model_rows(
    predictions: pd.DataFrame,
    *,
    panel: str,
    price_area: str,
    model_a: str,
    model_b: str,
) -> pd.DataFrame:
    key_columns = [
        "panel",
        "fold_id",
        "timestamp_utc",
        "price_area",
    ]

    selected_a = predictions.loc[
        predictions["panel"].eq(panel)
        & predictions["price_area"].eq(price_area)
        & predictions["model"].eq(model_a),
        [*key_columns, TARGET_COLUMN, "prediction"],
    ].rename(
        columns={
            TARGET_COLUMN: "actual_a",
            "prediction": "prediction_a",
        }
    )

    selected_b = predictions.loc[
        predictions["panel"].eq(panel)
        & predictions["price_area"].eq(price_area)
        & predictions["model"].eq(model_b),
        [*key_columns, TARGET_COLUMN, "prediction"],
    ].rename(
        columns={
            TARGET_COLUMN: "actual_b",
            "prediction": "prediction_b",
        }
    )

    if selected_a.duplicated(key_columns).any():
        raise RuntimeError(
            f"Duplicate prediction keys for {panel}/{price_area}/{model_a}."
        )

    if selected_b.duplicated(key_columns).any():
        raise RuntimeError(
            f"Duplicate prediction keys for {panel}/{price_area}/{model_b}."
        )

    paired = selected_a.merge(
        selected_b,
        on=key_columns,
        how="inner",
        validate="one_to_one",
    )

    paired = paired.replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna(
        subset=[
            "actual_a",
            "actual_b",
            "prediction_a",
            "prediction_b",
        ]
    )

    if paired.empty:
        return paired

    if not np.allclose(
        paired["actual_a"].to_numpy(dtype=float),
        paired["actual_b"].to_numpy(dtype=float),
        rtol=0.0,
        atol=1e-9,
    ):
        raise RuntimeError(
            f"Target mismatch on paired support for "
            f"{panel}/{price_area}/{model_a}/{model_b}."
        )

    paired["actual"] = paired["actual_a"].astype(float)
    paired["absolute_error_a"] = (
        paired["prediction_a"] - paired["actual"]
    ).abs()
    paired["absolute_error_b"] = (
        paired["prediction_b"] - paired["actual"]
    ).abs()
    paired["loss_difference_a_minus_b"] = (
        paired["absolute_error_a"]
        - paired["absolute_error_b"]
    )

    local_timestamp = (
        paired["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )
    paired["delivery_date_local"] = (
        local_timestamp.dt.normalize()
    )

    return paired.sort_values(
        ["timestamp_utc", "fold_id"]
    ).reset_index(drop=True)


def paired_accuracy_summary(
    paired: pd.DataFrame,
) -> dict[str, float]:
    if paired.empty:
        return {
            "common_observations": 0,
            "model_a_mae": np.nan,
            "model_b_mae": np.nan,
            "mae_difference_a_minus_b": np.nan,
            "relative_improvement_a_vs_b": np.nan,
            "hourly_win_rate_a": np.nan,
            "daily_win_rate_a": np.nan,
        }

    mae_a = float(paired["absolute_error_a"].mean())
    mae_b = float(paired["absolute_error_b"].mean())

    daily = (
        paired.groupby("delivery_date_local", as_index=False)
        .agg(
            absolute_error_a=("absolute_error_a", "mean"),
            absolute_error_b=("absolute_error_b", "mean"),
        )
    )

    return {
        "common_observations": int(len(paired)),
        "common_days": int(paired["delivery_date_local"].nunique()),
        "model_a_mae": mae_a,
        "model_b_mae": mae_b,
        "mae_difference_a_minus_b": mae_a - mae_b,
        "relative_improvement_a_vs_b": (
            (mae_b - mae_a) / mae_b
            if mae_b > 0
            else np.nan
        ),
        "hourly_win_rate_a": float(
            (paired["absolute_error_a"]
             < paired["absolute_error_b"]).mean()
        ),
        "daily_win_rate_a": float(
            (daily["absolute_error_a"]
             < daily["absolute_error_b"]).mean()
        ),
    }


def newey_west_mean_inference(
    values: pd.Series | np.ndarray,
    *,
    max_lag: int,
    alpha: float,
) -> dict[str, float]:
    x = _numeric_series(values).dropna().to_numpy(dtype=float)
    n = len(x)

    if n < 3:
        return {
            "hac_mean": np.nan,
            "hac_standard_error": np.nan,
            "hac_ci_lower": np.nan,
            "hac_ci_upper": np.nan,
            "hac_z_statistic": np.nan,
            "hac_p_value": np.nan,
            "hac_lag": 0,
        }

    mean_value = float(np.mean(x))
    centered = x - mean_value
    lag = min(int(max_lag), n - 1)

    gamma_zero = float(
        np.dot(centered, centered) / n
    )
    long_run_variance = gamma_zero

    for current_lag in range(1, lag + 1):
        weight = 1.0 - current_lag / (lag + 1.0)
        autocovariance = float(
            np.dot(
                centered[current_lag:],
                centered[:-current_lag],
            )
            / n
        )
        long_run_variance += (
            2.0 * weight * autocovariance
        )

    variance_of_mean = max(
        long_run_variance / n,
        0.0,
    )
    standard_error = float(np.sqrt(variance_of_mean))

    normal = NormalDist()
    critical_value = normal.inv_cdf(1.0 - alpha / 2.0)

    if standard_error > 0:
        z_statistic = mean_value / standard_error
        p_value = 2.0 * (
            1.0 - normal.cdf(abs(z_statistic))
        )
    else:
        z_statistic = np.nan
        p_value = np.nan

    return {
        "hac_mean": mean_value,
        "hac_standard_error": standard_error,
        "hac_ci_lower": (
            mean_value - critical_value * standard_error
        ),
        "hac_ci_upper": (
            mean_value + critical_value * standard_error
        ),
        "hac_z_statistic": float(z_statistic),
        "hac_p_value": float(p_value),
        "hac_lag": int(lag),
    }


def local_day_block_bootstrap_ci(
    paired: pd.DataFrame,
    *,
    draws: int,
    alpha: float,
    seed: int,
) -> dict[str, float]:
    if paired.empty:
        return {
            "bootstrap_ci_lower": np.nan,
            "bootstrap_ci_upper": np.nan,
            "bootstrap_draws": 0,
            "bootstrap_blocks": 0,
        }

    day_blocks = [
        group["loss_difference_a_minus_b"]
        .dropna()
        .to_numpy(dtype=float)
        for _, group in paired.groupby("delivery_date_local")
    ]
    day_blocks = [
        block for block in day_blocks if len(block) > 0
    ]

    if not day_blocks:
        return {
            "bootstrap_ci_lower": np.nan,
            "bootstrap_ci_upper": np.nan,
            "bootstrap_draws": 0,
            "bootstrap_blocks": 0,
        }

    rng = np.random.default_rng(seed)
    block_count = len(day_blocks)
    bootstrap_means = np.empty(draws, dtype=float)

    block_sums = np.array(
        [block.sum() for block in day_blocks],
        dtype=float,
    )
    block_sizes = np.array(
        [len(block) for block in day_blocks],
        dtype=float,
    )

    for draw in range(draws):
        sampled = rng.integers(
            0,
            block_count,
            size=block_count,
        )
        bootstrap_means[draw] = (
            block_sums[sampled].sum()
            / block_sizes[sampled].sum()
        )

    lower, upper = np.quantile(
        bootstrap_means,
        [alpha / 2.0, 1.0 - alpha / 2.0],
    )

    return {
        "bootstrap_ci_lower": float(lower),
        "bootstrap_ci_upper": float(upper),
        "bootstrap_draws": int(draws),
        "bootstrap_blocks": int(block_count),
    }


In [ ]:
# ============================================================
# 35A. ALL-MODEL COMMON-SUPPORT TABLES
# ============================================================

def all_pairwise_common_support_by_fold(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for keys, group in predictions.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys
        models = sorted(group["model"].dropna().unique())

        for model_a, model_b in combinations(models, 2):
            paired = paired_model_rows(
                predictions,
                panel=panel_name,
                price_area=price_area,
                model_a=model_a,
                model_b=model_b,
            )

            paired = paired.loc[
                paired["fold_id"].eq(fold_id)
            ].copy()

            if paired.empty:
                continue

            rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold_id,
                    "price_area": price_area,
                    "model_a": model_a,
                    "model_b": model_b,
                    **paired_accuracy_summary(paired),
                }
            )

    return pd.DataFrame(rows)


def common_support_vs_reference(
    predictions: pd.DataFrame,
    reference_models: Sequence[str],
) -> pd.DataFrame:
    rows = []

    for keys, group in predictions.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys
        models = sorted(group["model"].dropna().unique())

        for reference_model in reference_models:
            if reference_model not in models:
                continue

            for candidate_model in models:
                if candidate_model == reference_model:
                    continue

                paired = paired_model_rows(
                    predictions,
                    panel=panel_name,
                    price_area=price_area,
                    model_a=candidate_model,
                    model_b=reference_model,
                )

                paired = paired.loc[
                    paired["fold_id"].eq(fold_id)
                ].copy()

                if paired.empty:
                    continue

                rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold_id,
                        "price_area": price_area,
                        "candidate_model": candidate_model,
                        "reference_model": reference_model,
                        **paired_accuracy_summary(paired),
                    }
                )

    return pd.DataFrame(rows)


REFERENCE_MODELS = (
    "benchmark::energinet_wind_forecast_mw",
    "ecmwf",
    "pooled_lightgbm",
    "ensemble::equal_weight",
)

if RUN_MODE == "FULL":
    all_pairwise_common_support = (
        all_pairwise_common_support_by_fold(
            all_test_predictions
        )
    )

    common_support_reference = (
        common_support_vs_reference(
            all_test_predictions,
            REFERENCE_MODELS,
        )
    )

    all_pairwise_common_support.to_csv(
        TABLE_DIR
        / "notebook_02_all_pairwise_common_support_by_fold.csv",
        index=False,
    )

    common_support_reference.to_csv(
        TABLE_DIR
        / "notebook_02_common_support_vs_reference.csv",
        index=False,
    )

    display(
        common_support_reference.sort_values(
            [
                "panel",
                "price_area",
                "fold_id",
                "reference_model",
                "model_a_mae",
            ]
        ).head(100)
    )
else:
    all_pairwise_common_support = pd.DataFrame()
    common_support_reference = pd.DataFrame()


In [ ]:
# ============================================================
# 35B. PRE-SPECIFIED PAIRED INFERENCE
# ============================================================

CONFIRMATORY_COMPARISONS = (
    # D2 DK1
    ("d2", "DK1", "ensemble::linear_stack", "ecmwf"),
    ("d2", "DK1", "ensemble::linear_stack", "pooled_lightgbm"),
    (
        "d2",
        "DK1",
        "ensemble::linear_stack",
        "benchmark::energinet_wind_forecast_mw",
    ),
    # D2 DK2
    ("d2", "DK2", "pooled_lightgbm", "ecmwf"),
    ("d2", "DK2", "pooled_lightgbm", "ensemble::equal_weight"),
    (
        "d2",
        "DK2",
        "pooled_lightgbm",
        "benchmark::energinet_wind_forecast_mw",
    ),
    # Exact DK1
    ("exact", "DK1", "ensemble::dirichlet_mae", "ecmwf"),
    ("exact", "DK1", "ensemble::dirichlet_mae", "pooled_lightgbm"),
    (
        "exact",
        "DK1",
        "ensemble::dirichlet_mae",
        "ensemble::equal_weight",
    ),
    # Exact DK2
    ("exact", "DK2", "pooled_lightgbm", "ecmwf"),
    (
        "exact",
        "DK2",
        "pooled_lightgbm",
        "ensemble::inverse_mae",
    ),
    (
        "exact",
        "DK2",
        "pooled_lightgbm",
        "ensemble::equal_weight",
    ),
)


def confirmatory_pairwise_report(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for comparison_number, (
        panel_name,
        price_area,
        model_a,
        model_b,
    ) in enumerate(CONFIRMATORY_COMPARISONS, start=1):
        paired = paired_model_rows(
            predictions,
            panel=panel_name,
            price_area=price_area,
            model_a=model_a,
            model_b=model_b,
        )

        summary = paired_accuracy_summary(paired)
        hac = newey_west_mean_inference(
            paired.get(
                "loss_difference_a_minus_b",
                pd.Series(dtype=float),
            ),
            max_lag=HAC_MAX_LAG_HOURS,
            alpha=PAIRWISE_ALPHA,
        )
        bootstrap = local_day_block_bootstrap_ci(
            paired,
            draws=BLOCK_BOOTSTRAP_DRAWS,
            alpha=PAIRWISE_ALPHA,
            seed=stable_seed(
                RANDOM_SEED,
                panel_name,
                price_area,
                model_a,
                model_b,
                comparison_number,
            ),
        )

        rows.append(
            {
                "panel": panel_name,
                "price_area": price_area,
                "model_a": model_a,
                "model_b": model_b,
                "interpretation": (
                    "negative difference favours model_a"
                ),
                **summary,
                **hac,
                **bootstrap,
                "hac_significant_at_5pct": bool(
                    np.isfinite(hac["hac_ci_lower"])
                    and np.isfinite(hac["hac_ci_upper"])
                    and (
                        hac["hac_ci_upper"] < 0
                        or hac["hac_ci_lower"] > 0
                    )
                ),
                "bootstrap_significant_at_5pct": bool(
                    np.isfinite(bootstrap["bootstrap_ci_lower"])
                    and np.isfinite(bootstrap["bootstrap_ci_upper"])
                    and (
                        bootstrap["bootstrap_ci_upper"] < 0
                        or bootstrap["bootstrap_ci_lower"] > 0
                    )
                ),
            }
        )

    return pd.DataFrame(rows)


if RUN_MODE == "FULL":
    confirmatory_pairwise_inference = (
        confirmatory_pairwise_report(
            all_test_predictions
        )
    )

    confirmatory_pairwise_inference.to_csv(
        TABLE_DIR
        / "notebook_02_confirmatory_pairwise_inference.csv",
        index=False,
    )

    display(confirmatory_pairwise_inference)
else:
    confirmatory_pairwise_inference = pd.DataFrame()


In [ ]:
# ============================================================
# 35C. D2 AND EXACT COMPARISON ON COMMON DELIVERY TIMESTAMPS
# ============================================================

def unique_track_predictions(
    predictions: pd.DataFrame,
    *,
    panel_name: str,
    model_name: str,
    price_area: str,
) -> pd.DataFrame:
    selected = predictions.loc[
        predictions["panel"].eq(panel_name)
        & predictions["model"].eq(model_name)
        & predictions["price_area"].eq(price_area),
        [
            "timestamp_utc",
            "price_area",
            TARGET_COLUMN,
            "prediction",
        ],
    ].copy()

    key_columns = ["timestamp_utc", "price_area"]

    if selected.duplicated(key_columns).any():
        raise RuntimeError(
            f"Repeated delivery timestamps remain for "
            f"{panel_name}/{price_area}/{model_name}."
        )

    return selected


def d2_exact_overlap_report(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    common_models = sorted(
        set(
            predictions.loc[
                predictions["panel"].eq("d2"),
                "model",
            ]
        )
        & set(
            predictions.loc[
                predictions["panel"].eq("exact"),
                "model",
            ]
        )
    )

    for price_area in PRICE_AREAS:
        for model_name in common_models:
            d2_selected = unique_track_predictions(
                predictions,
                panel_name="d2",
                model_name=model_name,
                price_area=price_area,
            ).rename(
                columns={
                    TARGET_COLUMN: "actual_d2",
                    "prediction": "prediction_d2",
                }
            )

            exact_selected = unique_track_predictions(
                predictions,
                panel_name="exact",
                model_name=model_name,
                price_area=price_area,
            ).rename(
                columns={
                    TARGET_COLUMN: "actual_exact",
                    "prediction": "prediction_exact",
                }
            )

            overlap = d2_selected.merge(
                exact_selected,
                on=["timestamp_utc", "price_area"],
                how="inner",
                validate="one_to_one",
            ).dropna()

            if overlap.empty:
                continue

            if not np.allclose(
                overlap["actual_d2"].to_numpy(dtype=float),
                overlap["actual_exact"].to_numpy(dtype=float),
                rtol=0.0,
                atol=1e-9,
            ):
                raise RuntimeError(
                    f"D2/exact target mismatch for "
                    f"{price_area}/{model_name}."
                )

            d2_metrics = safe_regression_metrics(
                overlap["actual_d2"],
                overlap["prediction_d2"],
            )
            exact_metrics = safe_regression_metrics(
                overlap["actual_exact"],
                overlap["prediction_exact"],
            )

            rows.append(
                {
                    "price_area": price_area,
                    "model": model_name,
                    "common_observations": int(len(overlap)),
                    "d2_mae": d2_metrics["mae"],
                    "exact_mae": exact_metrics["mae"],
                    "exact_minus_d2_mae": (
                        exact_metrics["mae"]
                        - d2_metrics["mae"]
                    ),
                    "d2_rmse": d2_metrics["rmse"],
                    "exact_rmse": exact_metrics["rmse"],
                    "d2_bias": d2_metrics["bias"],
                    "exact_bias": exact_metrics["bias"],
                }
            )

    return pd.DataFrame(rows)


if RUN_MODE == "FULL":
    overlap_comparison = d2_exact_overlap_report(
        all_test_predictions
    )

    overlap_comparison.to_csv(
        TABLE_DIR
        / "notebook_02_d2_exact_common_timestamp_comparison.csv",
        index=False,
    )

    display(
        overlap_comparison.sort_values(
            ["price_area", "model"]
        )
    )
else:
    overlap_comparison = pd.DataFrame()


## Verification gate

`READY_FOR_FULL_MODELLING` requires:

- both frozen panels to match their expected shapes;
- zero unresolved manual-review features;
- valid time-aware folds;
- successful baseline generation.

`MODELLING_COMPLETE` additionally requires non-empty provider-expert, static-weight, Hedge, regime-weight, pooled-model, and metric outputs for both the D2 and exact-run tracks.

In [ ]:
# ============================================================
# 36. FINAL VERIFICATION
# ============================================================

blocking_failures: list[str] = []

if d2_panel.shape != EXPECTED_D2_SHAPE:
    blocking_failures.append(
        f"D2 shape mismatch: {d2_panel.shape}"
    )

if exact_panel.shape != EXPECTED_EXACT_SHAPE:
    blocking_failures.append(
        f"Exact shape mismatch: {exact_panel.shape}"
    )

if temporal_folds.empty:
    blocking_failures.append(
        "Temporal fold manifest is empty."
    )

if int(test_overlap_summary["overlapping_test_keys"].sum()) != 0:
    blocking_failures.append(
        "Outer test windows overlap across folds."
    )

remaining_manual_review = pd.concat(
    [
        d2_feature_registry.loc[
            d2_feature_registry[
                "decision_time_status"
            ].eq("manual_review")
        ],
        exact_feature_registry.loc[
            exact_feature_registry[
                "decision_time_status"
            ].eq("manual_review")
        ],
    ],
    ignore_index=True,
)

if not remaining_manual_review.empty:
    blocking_failures.append(
        "Manual-review features remain unresolved."
    )

climatology_rows = baseline_predictions.loc[
    baseline_predictions["model"].eq(
        "hourly_climatology"
    )
]

if climatology_rows.empty:
    blocking_failures.append(
        "Hourly climatology output is empty."
    )
elif climatology_rows["prediction"].isna().any():
    blocking_failures.append(
        "Hourly climatology still contains missing predictions."
    )

if not residual_correlations.empty:
    invalid_residual_pairs = []

    for row in residual_correlations.itertuples(index=False):
        allowed = set(PANEL_PROVIDER_MAP[row.panel])

        if (
            row.expert_a not in allowed
            or row.expert_b not in allowed
        ):
            invalid_residual_pairs.append(
                (
                    row.panel,
                    row.expert_a,
                    row.expert_b,
                )
            )

    if invalid_residual_pairs:
        blocking_failures.append(
            "Residual-correlation output contains invalid experts."
        )

if RUN_MODE == "FULL":
    required_full_outputs = {
        "expert_predictions": expert_predictions,
        "static_ensemble_weights": ensemble_weights,
        "static_ensemble_predictions": static_ensemble_predictions,
        "stacking_predictions": stacking_predictions,
        "hedge_predictions": hedge_prediction_frame,
        "hedge_reports": hedge_reports,
        "regime_predictions": regime_ensemble_predictions,
        "regime_weights": regime_ensemble_weights,
        "pooled_predictions": pooled_predictions,
        "model_metrics": model_metrics,
        "all_pairwise_common_support": all_pairwise_common_support,
        "common_support_reference": common_support_reference,
        "confirmatory_pairwise_inference": (
            confirmatory_pairwise_inference
        ),
        "d2_exact_overlap_comparison": overlap_comparison,
    }

    for output_name, output_frame in required_full_outputs.items():
        if output_frame.empty:
            blocking_failures.append(
                f"Full-mode output is empty: {output_name}"
            )

    observed_full_panels = set(
        expert_predictions["panel"].unique()
    ) if not expert_predictions.empty else set()

    if observed_full_panels != {"d2", "exact"}:
        blocking_failures.append(
            "Full modelling did not produce both d2 and exact tracks."
        )

if blocking_failures:
    verification_status = "BLOCKED"
elif RUN_MODE == "AUDIT":
    verification_status = "READY_FOR_FULL_MODELLING"
else:
    verification_status = "MODELLING_COMPLETE"

verification = {
    "verification_status": verification_status,
    "run_mode": RUN_MODE,
    "search_mode": SEARCH_MODE,
    "output_run_name": OUTPUT_RUN_NAME,
    "blocking_failures": blocking_failures,
    "d2_shape": list(d2_panel.shape),
    "exact_shape": list(exact_panel.shape),
    "d2_folds": len(d2_folds),
    "exact_folds": len(exact_folds),
    "d2_fold_config": D2_FOLD_CONFIG,
    "exact_fold_config": EXACT_FOLD_CONFIG,
    "overlapping_test_keys": int(
        test_overlap_summary[
            "overlapping_test_keys"
        ].sum()
    ),
    "manual_review_feature_count": int(
        len(remaining_manual_review)
    ),
    "d2_used": True,
    "exact_used": True,
    "baseline_prediction_rows": int(
        len(baseline_predictions)
    ),
    "all_test_prediction_rows": int(
        len(all_test_predictions)
    ),
    "climatology_missing_predictions": int(
        climatology_rows["prediction"].isna().sum()
    ),
    "confirmatory_comparisons": int(
        len(confirmatory_pairwise_inference)
        if RUN_MODE == "FULL"
        else 0
    ),
    "lightgbm_device_type": LIGHTGBM_BASE_PARAMS.get(
        "device_type",
        "cpu",
    ),
    "lightgbm_gpu_platform_id": LIGHTGBM_BASE_PARAMS.get(
        "gpu_platform_id",
    ),
    "lightgbm_gpu_device_id": LIGHTGBM_BASE_PARAMS.get(
        "gpu_device_id",
    ),
    "lightgbm_gpu_use_dp": LIGHTGBM_BASE_PARAMS.get(
        "gpu_use_dp",
    ),
    "lightgbm_max_bin": LIGHTGBM_BASE_PARAMS.get(
        "max_bin",
    ),
    "algorithms": [
        "provider_lightgbm",
        "pooled_lightgbm",
        "equal_weight",
        "best_validation_expert",
        "inverse_mae",
        "dirichlet_mae",
        "slsqp_mae",
        "slsqp_mse",
        "slsqp_mae_median_intercept",
        "linear_stack",
        "forecast_origin_safe_hedge",
        "forecast_disagreement_regime_slsqp_mae",
        "paired_common_support_evaluation",
        "newey_west_inference",
        "local_day_block_bootstrap",
    ],
    "input_manifest": dataset_manifest,
}

verification_path = (
    AUDIT_DIR
    / "notebook_02_verification.json"
)

with verification_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        verification,
        file,
        indent=2,
        default=str,
    )

print("=== NOTEBOOK 02 VERIFICATION ===")
print(
    json.dumps(
        verification,
        indent=2,
        default=str,
    )
)


## Outputs

All v5 artifacts are written under:

```text
outputs/notebook_02_v5_full/
```

Key outputs include:

```text
audits/notebook_02_verification.json
audits/notebook_02_test_window_overlap_audit.csv
audits/notebook_02_expert_residual_correlations.csv

tables/notebook_02_model_metrics_by_fold.csv
tables/notebook_02_model_metric_summary.csv
tables/notebook_02_common_support_vs_reference.csv
tables/notebook_02_all_pairwise_common_support_by_fold.csv
tables/notebook_02_confirmatory_pairwise_inference.csv
tables/notebook_02_d2_exact_common_timestamp_comparison.csv

predictions/notebook_02_test_predictions.parquet
```

A negative paired loss difference means `model_a` has lower MAE than `model_b`.
